# Emulators + LOOCV for $v_0$ 

In [3]:
#!/usr/bin/env python3
"""
Emulator training + LOO validation for global v0.
"""

import os
import time
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

# =============================================================================
# CONFIGURATION
# =============================================================================

PCA_RESULTS_FILE = "pca_results/pca_results_global_v0_all_corrections.pkl"
DESIGN_FILE      = "design_pts_Pb_Pb_2760_production/design_points_main_PbPb-2760.dat"
RANGE_FILE       = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"

EMULATOR_FOLDER   = "emulators_global_v0"
VALIDATION_FOLDER = "emulator_validation_global_v0"

N_PC_TRAIN = None   # None = all PCs
RETRAIN    = False

# ------------------------------------------------------------------
# BAD DESIGN POINTS
# ------------------------------------------------------------------
NAN_SETS_BY_IDF = {
    0: {334, 341, 377, 429, 447, 483},
    1: {285, 334, 341, 447, 483, 495},
    2: {209, 280, 322, 334, 341, 412, 421, 424, 429, 432, 446, 447, 453, 468, 483, 495},
    3: {60, 232, 280, 285, 322, 324, 341, 377, 432, 447, 464, 468, 482, 483, 485, 495},
}
UNFINISHED_EVENTS_SET = {289, 324, 326, 459, 462, 242, 406, 440, 123}
STRANGE_FEATURES_SET  = {289, 324, 440, 459, 462}

CORRECTION_TO_IDF = {'Grad': 0, 'CE': 1, 'PTM': 2, 'PTB': 3}

# ------------------------------------------------------------------
# FIGURE settings
# ------------------------------------------------------------------
SAVE_FIGURES  = True
FIGURE_FORMAT = 'pdf'
FIGURE_DPI    = 300

# =============================================================================
# UTILITY
# =============================================================================

def save_fig(fig, filename, folder, fmt=FIGURE_FORMAT, dpi=FIGURE_DPI):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    if fmt == 'both':
        for ext in ('pdf', 'png'):
            path = folder / f'{stem}.{ext}'
            fig.savefig(path, dpi=dpi, bbox_inches='tight')
            print(f"    Saved: {path}")
    else:
        path = folder / f'{stem}.{fmt}'
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print(f"    Saved: {path}")
    plt.close(fig)


def build_bad_points_set(idf):
    return NAN_SETS_BY_IDF[idf] | UNFINISHED_EVENTS_SET | STRANGE_FEATURES_SET


def align_design_to_pca(design_df, design_point_ids):
    return design_df.iloc[design_point_ids].values


def r2_score(y_true, y_pred):
    """R² that never crashes on zero-variance columns."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

# =============================================================================
# LOAD DESIGN POINTS
# =============================================================================

print("\n" + "="*80)
print("EMULATOR TRAINING FOR GLOBAL v0  —  with LOO validation")
print("="*80)

print("\nLoading design points...")
design_df     = pd.read_csv(DESIGN_FILE, index_col=0)
design_range  = pd.read_csv(RANGE_FILE,  index_col=0)
design_max    = design_range["max"].values
design_min    = design_range["min"].values
design_labels = list(design_df.columns)
print(f"  ✓ {design_df.shape[0]} design points, {design_df.shape[1]} parameters")

# =============================================================================
# LOAD PCA RESULTS
# =============================================================================

print("\nLoading PCA results...")
if not Path(PCA_RESULTS_FILE).exists():
    raise FileNotFoundError(f"PCA file not found: {PCA_RESULTS_FILE}\n"
                            "Run pca_global_v0.py first.")

with open(PCA_RESULTS_FILE, "rb") as f:
    pca_results_dict = pickle.load(f)

print("  ✓ PCA results loaded:")
for k, v in pca_results_dict.items():
    print(f"    • {k}: {v['n_components']} PCs, "
          f"{len(v['design_point_ids'])} design points")

Path(EMULATOR_FOLDER).mkdir(parents=True, exist_ok=True)
Path(VALIDATION_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# KERNEL BUILDER
# =============================================================================

def make_kernel(X):
    ptp = np.ptp(X, axis=0)
    ptp[ptp == 0] = 1.0
    return (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=ptp,
              length_scale_bounds=np.outer(ptp, (0.4, 1e2)))
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-2, 1e2))
    )

# =============================================================================
# TRAINING
# =============================================================================

def train_emulators(X, pc_scores, n_pc_train=None,
                    emulator_file=None, retrain=False):
    if n_pc_train is None:
        n_pc_train = pc_scores.shape[1]
    n_pc_train = min(n_pc_train, pc_scores.shape[1])

    if emulator_file and Path(emulator_file).exists() and not retrain:
        print(f"  Loading existing emulators from {emulator_file}")
        with open(emulator_file, "rb") as f:
            emulators = pickle.load(f)
        # Trim to n_pc_train in case the saved file has more
        return emulators[:n_pc_train]

    print(f"  Training {n_pc_train} GP emulators...")
    emulators = []
    for i in range(n_pc_train):
        t0  = time.time()
        gpr = GaussianProcessRegressor(
            kernel=make_kernel(X),
            n_restarts_optimizer=5,
            alpha=1e-10,
            normalize_y=True,
        )
        gpr.fit(X, pc_scores[:, i])
        r2 = gpr.score(X, pc_scores[:, i])
        print(f"    PC {i+1:02d}: R²={r2:.4f}  ({time.time()-t0:.1f}s)")
        emulators.append(gpr)

    if emulator_file:
        with open(emulator_file, "wb") as f:
            pickle.dump(emulators, f)
        print(f"  Saved → {emulator_file}")

    return emulators

# =============================================================================
# LOO VALIDATION  (fast analytical)
# =============================================================================

def loo_validate_fast(X, pc_scores, emulators):
    """
    Analytical LOO using the GP hat-matrix formula.
    Exact for fixed hyperparameters; runs in seconds.

    Returns
    -------
    y_true, y_pred, y_std  — each (n, n_pc)
    """
    n    = X.shape[0]
    n_pc = min(len(emulators), pc_scores.shape[1])   # key guard

    y_true = pc_scores[:, :n_pc].copy()
    y_pred = np.zeros_like(y_true)
    y_std  = np.zeros_like(y_true)

    for i, gpr in enumerate(emulators[:n_pc]):        # iterate safely
        K       = gpr.kernel_(X)
        noise   = np.exp(gpr.kernel_.theta[-1])
        K_noisy = K + noise * np.eye(n)

        L     = np.linalg.cholesky(K_noisy)
        L_inv = np.linalg.solve(L, np.eye(n))
        K_inv = L_inv.T @ L_inv

        y = y_true[:, i]
        if gpr.normalize_y:
            y_mean = y.mean()
            y_std_ = y.std() or 1.0
            y_norm = (y - y_mean) / y_std_
        else:
            y_norm = y
            y_mean, y_std_ = 0.0, 1.0

        diag         = K_inv.diagonal()
        loo_norm     = y_norm - (K_inv @ y_norm) / (diag + 1e-12)
        loo_var      = 1.0 / (diag + 1e-12)

        y_pred[:, i] = loo_norm * y_std_ + y_mean
        y_std[:, i]  = np.sqrt(np.abs(loo_var)) * y_std_

    return y_true, y_pred, y_std

# =============================================================================
# HELPER: per-PC LOO R² list
# =============================================================================

def loo_r2_list(y_true, y_pred):
    """Return list of LOO R² values, one per PC column."""
    n_pc = y_true.shape[1]
    return [r2_score(y_true[:, i], y_pred[:, i]) for i in range(n_pc)]

# =============================================================================
# PLOTS
# =============================================================================

def plot_loo_per_pc(y_true, y_pred, y_std, correction_name, save_folder):
    n_pc  = y_true.shape[1]
    ncols = min(4, n_pc)
    nrows = math.ceil(n_pc / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.5 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i in range(n_pc):
        ax = axes[i]
        yt = y_true[:, i]
        yp = y_pred[:, i]
        ys = y_std[:, i]

        z  = np.abs((yt - yp) / (ys + 1e-12))
        sc = ax.scatter(yt, yp, c=z, cmap='RdYlGn_r',
                        vmin=0, vmax=3, s=18, alpha=0.7)
        plt.colorbar(sc, ax=ax, label='|z|')

        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'k--', lw=1.5)

        r2   = r2_score(yt, yp)
        rmse = np.sqrt(np.mean((yt - yp) ** 2))
        c68  = np.mean(np.abs(yt - yp) < ys) * 100
        c95  = np.mean(np.abs(yt - yp) < 2 * ys) * 100

        ax.set_title(f"PC {i+1} | R²={r2:.3f}  RMSE={rmse:.3f}",
                     fontsize=10, fontweight='bold')
        ax.set_xlabel("True PC score", fontsize=9)
        ax.set_ylabel("LOO predicted", fontsize=9)
        ax.text(0.04, 0.96,
                f"68% cov: {c68:.0f}%\n95% cov: {c95:.0f}%",
                transform=ax.transAxes, fontsize=8,
                va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
        ax.grid(True, alpha=0.3)

    for j in range(n_pc, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'LOO Validation — {correction_name} — Global $v_0$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'loo_validation_global_v0_{correction_name}', save_folder)


def plot_loo_residuals(y_true, y_pred, y_std, correction_name, save_folder):
    n_pc  = y_true.shape[1]
    ncols = min(4, n_pc)
    nrows = math.ceil(n_pc / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.0 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    x_ref = np.linspace(-5, 5, 300)
    gauss = np.exp(-0.5 * x_ref**2) / np.sqrt(2 * np.pi)

    for i in range(n_pc):
        ax = axes[i]
        z  = (y_true[:, i] - y_pred[:, i]) / (y_std[:, i] + 1e-12)

        ax.hist(z, bins=30, density=True, color='steelblue',
                alpha=0.6, label='LOO pulls')
        ax.plot(x_ref, gauss, 'r-', lw=2, label='N(0,1)')
        ax.set_xlim(-5, 5)
        ax.set_xlabel(r'Pull  $(y - \hat{y})\,/\,\sigma$', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'PC {i+1}  |  '
                     f'$\\mu$={z.mean():.2f}  $\\sigma$={z.std():.2f}',
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for j in range(n_pc, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'LOO Pull Distributions — {correction_name} — Global $v_0$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'loo_pulls_global_v0_{correction_name}', save_folder)


def plot_summary(all_emulators, all_loo, pca_results_dict,
                 all_design_matrices, save_folder):
    corrections = list(all_emulators.keys())
    colors      = plt.cm.tab10(np.linspace(0, 1, len(corrections)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for corr, color in zip(corrections, colors):
        emulators = all_emulators[corr]
        X         = all_design_matrices[corr]
        pc_scores = pca_results_dict[corr]['pc_scores']

        # Safe n_pc: never exceed either emulators or pc_scores columns
        n_pc = min(len(emulators), pc_scores.shape[1])

        # In-sample R²
        r2_in = [r2_score(pc_scores[:, i], emulators[i].predict(X))
                 for i in range(n_pc)]
        axes[0].plot(range(1, n_pc + 1), r2_in, 'o-',
                     color=color, label=corr, lw=2, markersize=6)

        # LOO R²
        r2_loo = loo_r2_list(all_loo[corr]['y_true'],
                              all_loo[corr]['y_pred'])
        axes[1].plot(range(1, len(r2_loo) + 1), r2_loo, 'o-',
                     color=color, label=corr, lw=2, markersize=6)

    for ax, title in zip(axes, ['In-sample R²', 'LOO R²']):
        ax.set_xlabel('Principal Component', fontsize=12)
        ax.set_ylabel('R²', fontsize=12)
        ax.set_title(f'{title} — Global $v_0$', fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.5, 1.02)

    plt.tight_layout()
    save_fig(fig, 'emulator_summary_global_v0', save_folder)

# =============================================================================
# MAIN LOOP
# =============================================================================

all_emulators       = {}
all_loo             = {}
all_design_matrices = {}

for corr, pca_res in pca_results_dict.items():
    print("\n" + "="*80)
    print(f"Viscous correction: {corr}")
    print("="*80)

    pc_scores        = pca_res['pc_scores']
    design_point_ids = pca_res['design_point_ids']
    n_pc_total       = pca_res['n_components']

    X = align_design_to_pca(design_df, design_point_ids)
    all_design_matrices[corr] = X

    print(f"  Design matrix shape : {X.shape}")
    print(f"  PC scores shape     : {pc_scores.shape}")

    idf     = CORRECTION_TO_IDF[corr]
    bad_pts = build_bad_points_set(idf)
    overlap = set(design_point_ids) & bad_pts
    if overlap:
        print(f"  ⚠️  WARNING: {len(overlap)} bad points still present: {overlap}")
    else:
        print(f"  ✓ No bad design points in this dataset")

    # 1. Train
    print(f"\n1. Training emulators...")
    n_pc_train    = N_PC_TRAIN if N_PC_TRAIN else n_pc_total
    emulator_file = f"{EMULATOR_FOLDER}/gp_emulators_global_v0_{corr}.pkl"

    emulators = train_emulators(
        X, pc_scores,
        n_pc_train    = n_pc_train,
        emulator_file = emulator_file,
        retrain       = RETRAIN,
    )
    all_emulators[corr] = emulators

    # 2. LOO
    print(f"\n2. LOO validation (fast analytical)...")
    y_true, y_pred, y_std = loo_validate_fast(X, pc_scores, emulators)
    all_loo[corr] = {'y_true': y_true, 'y_pred': y_pred, 'y_std': y_std}

    # 3. Per-PC metrics
    n_pc = y_true.shape[1]
    print(f"\n   {'PC':>4}  {'R²(LOO)':>9}  {'RMSE':>8}  {'68% cov':>9}  {'95% cov':>9}")
    print(f"   {'-'*4}  {'-'*9}  {'-'*8}  {'-'*9}  {'-'*9}")
    for i in range(n_pc):
        yt = y_true[:, i]; yp = y_pred[:, i]; ys = y_std[:, i]
        r2   = r2_score(yt, yp)
        rmse = np.sqrt(np.mean((yt - yp) ** 2))
        c68  = np.mean(np.abs(yt - yp) < ys) * 100
        c95  = np.mean(np.abs(yt - yp) < 2 * ys) * 100
        print(f"   {i+1:>4}  {r2:>9.4f}  {rmse:>8.4f}  {c68:>8.1f}%  {c95:>8.1f}%")

    # 4. Plots
    print(f"\n3. Saving validation plots...")
    corr_val_folder = f"{VALIDATION_FOLDER}/{corr}"
    plot_loo_per_pc(y_true, y_pred, y_std, corr, corr_val_folder)
    plot_loo_residuals(y_true, y_pred, y_std, corr, corr_val_folder)

    print(f"\n✓ Finished {corr}")

# =============================================================================
# SUMMARY PLOT
# =============================================================================

print("\n" + "="*80)
print("SUMMARY PLOT")
plot_summary(all_emulators, all_loo, pca_results_dict,
             all_design_matrices, VALIDATION_FOLDER)

# =============================================================================
# SAVE ALL EMULATORS
# =============================================================================

print("\n" + "="*80)
print("SAVING ALL EMULATORS")
emulator_file_all = f"{EMULATOR_FOLDER}/all_gp_emulators_global_v0.pkl"
with open(emulator_file_all, "wb") as f:
    pickle.dump(all_emulators, f)
print(f"  ✓ {emulator_file_all}")

# =============================================================================
# FINAL SUMMARY TABLE
# =============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY — GLOBAL v0")
print("="*80)
print(f"\n{'Correction':<10} {'PCs':>5} {'DPs':>6} "
      f"{'min R²(in)':>12} {'min R²(LOO)':>13} {'avg R²(LOO)':>13}")
print("-" * 65)

for corr in all_emulators:
    emulators = all_emulators[corr]
    X         = all_design_matrices[corr]
    pc_scores = pca_results_dict[corr]['pc_scores']

    # Safe n_pc
    n_pc = min(len(emulators), pc_scores.shape[1])

    r2_in  = [r2_score(pc_scores[:, i], emulators[i].predict(X))
              for i in range(n_pc)]
    r2_loo = loo_r2_list(all_loo[corr]['y_true'],
                          all_loo[corr]['y_pred'])

    print(f"{corr:<10} {n_pc:>5} {X.shape[0]:>6} "
          f"{min(r2_in):>12.4f} {min(r2_loo):>13.4f} {np.nanmean(r2_loo):>13.4f}")

print("\n" + "="*80)
print("✓ Done")
print(f"  Emulators : {EMULATOR_FOLDER}/")
print(f"  Validation: {VALIDATION_FOLDER}/")
print("="*80 + "\n")


EMULATOR TRAINING FOR GLOBAL v0  —  with LOO validation

Loading design points...
  ✓ 500 design points, 17 parameters

Loading PCA results...
  ✓ PCA results loaded:
    • Grad: 3 PCs, 485 design points
    • CE: 3 PCs, 485 design points
    • PTM: 3 PCs, 475 design points
    • PTB: 3 PCs, 476 design points

Viscous correction: Grad
  Design matrix shape : (485, 17)
  PC scores shape     : (485, 3)
  ✓ No bad design points in this dataset

1. Training emulators...
  Loading existing emulators from emulators_global_v0/gp_emulators_global_v0_Grad.pkl

2. LOO validation (fast analytical)...

     PC    R²(LOO)      RMSE    68% cov    95% cov
   ----  ---------  --------  ---------  ---------
      1     0.9560    0.2095      82.9%      97.9%
      2     0.8885    0.3335      83.3%      97.7%
      3     0.6563    0.5856      79.6%      97.7%

3. Saving validation plots...
    Saved: emulator_validation_global_v0/Grad/loo_validation_global_v0_Grad.pdf
    Saved: emulator_validation_glob

# GSA for global $v_0$ 

In [4]:
#!/usr/bin/env python3
"""
Global Sobol Sensitivity Analysis for global v0 — all viscous corrections.
"""

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from SALib.sample import saltelli
from SALib.analyze import sobol

# =============================================================================
# CONFIGURATION
# =============================================================================

PCA_RESULTS_FILE = "pca_results/pca_results_global_v0_all_corrections.pkl"
EMULATOR_FOLDER  = "emulators_global_v0"
OUTPUT_FOLDER    = "GSA_global_v0"

DESIGN_FILE = "design_pts_Pb_Pb_2760_production/design_points_main_PbPb-2760.dat"
RANGE_FILE  = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"

N_SOBOL_SAMPLES = 1024
N_PC_ANALYZE    = None   # None = all PCs

VISCOUS_CORRECTIONS = ["Grad", "CE", "PTM", "PTB"]

# ------------------------------------------------------------------
# BAD DESIGN POINTS
# ------------------------------------------------------------------
NAN_SETS_BY_IDF = {
    0: {334, 341, 377, 429, 447, 483},
    1: {285, 334, 341, 447, 483, 495},
    2: {209, 280, 322, 334, 341, 412, 421, 424, 429, 432, 446, 447, 453, 468, 483, 495},
    3: {60, 232, 280, 285, 322, 324, 341, 377, 432, 447, 464, 468, 482, 483, 485, 495},
}
UNFINISHED_EVENTS_SET = {289, 324, 326, 459, 462, 242, 406, 440, 123}
STRANGE_FEATURES_SET  = {289, 324, 440, 459, 462}
CORRECTION_TO_IDF     = {'Grad': 0, 'CE': 1, 'PTM': 2, 'PTB': 3}

# ------------------------------------------------------------------
# FIGURE settings
# ------------------------------------------------------------------
FIGURE_FORMAT = 'pdf'
FIGURE_DPI    = 300

# =============================================================================
# PARAMETER NAMES
# =============================================================================

MODEL_PARAM_NAMES = [
    '$N$[$2.76$TeV]', '$p$', '$\\sigma_k$', '$w$ [fm]',
    '$d_{\\mathrm{min}}$ [fm]', '$\\tau_R$ [fm/$c$]', '$\\alpha$',
    '$T_{\\eta,\\mathrm{kink}}$ [GeV]',
    '$a_{\\eta,\\mathrm{low}}$ [GeV${}^{-1}$]',
    '$a_{\\eta,\\mathrm{high}}$ [GeV${}^{-1}$]',
    '$(\\eta/s)_{\\mathrm{kink}}$', '$(\\zeta/s)_{\\max}$',
    '$T_{\\zeta,c}$ [GeV]', '$w_{\\zeta}$ [GeV]',
    '$\\lambda_{\\zeta}$', '$b_{\\pi}$', '$T_{\\mathrm{sw}}$ [GeV]'
]

MODEL_PARAM_NAMES_SHORT = [
    'N', 'p', 'σ_k', 'w', 'd_min', 'τ_R', 'α',
    'T_η,kink', 'a_η,low', 'a_η,high', '(η/s)_kink',
    '(ζ/s)_max', 'T_ζ,c', 'w_ζ', 'λ_ζ', 'b_π', 'T_sw'
]

# =============================================================================
# UTILITY
# =============================================================================

def save_fig(fig, path, fmt=FIGURE_FORMAT, dpi=FIGURE_DPI):
    stem = Path(path).stem
    folder = Path(path).parent
    if fmt == 'both':
        for ext in ('pdf', 'png'):
            p = folder / f'{stem}.{ext}'
            fig.savefig(p, dpi=dpi, bbox_inches='tight')
            print(f"    Saved: {p}")
    else:
        p = folder / f'{stem}.{fmt}'
        fig.savefig(p, dpi=dpi, bbox_inches='tight')
        print(f"    Saved: {p}")
    plt.close(fig)


def build_bad_points_set(idf):
    return NAN_SETS_BY_IDF[idf] | UNFINISHED_EVENTS_SET | STRANGE_FEATURES_SET


def safe_n_pc(emulators, pca_res, n_pc_analyze):
    """Return the number of PCs to actually use, guarding all three limits."""
    return min(
        len(emulators),
        pca_res['n_components'],
        n_pc_analyze if n_pc_analyze is not None else 999
    )

# =============================================================================
# LOAD DATA
# =============================================================================

print("\n" + "="*80)
print("GLOBAL SENSITIVITY ANALYSIS FOR GLOBAL v0")
print("="*80)

print("\nLoading design parameter ranges...")
design_df    = pd.read_csv(DESIGN_FILE, index_col=0)
design_range = pd.read_csv(RANGE_FILE,  index_col=0)
design_max   = design_range["max"].values
design_min   = design_range["min"].values
print(f"  ✓ {len(design_min)} parameters")

print("\nLoading PCA results...")
if not Path(PCA_RESULTS_FILE).exists():
    raise FileNotFoundError(f"PCA file not found: {PCA_RESULTS_FILE}")

with open(PCA_RESULTS_FILE, "rb") as f:
    pca_results = pickle.load(f)

print("  ✓ PCA results loaded:")
for k, v in pca_results.items():
    n_removed = len(v.get('removed_design_points', []))
    print(f"    • {k}: {v['n_components']} PCs, "
          f"{len(v['design_point_ids'])} DPs used "
          f"({n_removed} removed)")

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# SOBOL ANALYSIS
# =============================================================================

def perform_sobol_analysis(emulators, n_pc, design_min, design_max,
                           param_names, n_samples, save_folder, correction_name):
    """
    Sobol sensitivity analysis using the trained emulators.
    Saltelli samples are drawn from the full parameter space —
    bad design points only affect training, not sampling here.
    """
    n_params = len(design_min)

    problem = {
        "num_vars": n_params,
        "names":    param_names,
        "bounds":   list(zip(design_min, design_max))
    }

    print(f"  Generating Saltelli samples (N={n_samples})...")
    X_sobol = saltelli.sample(problem, n_samples, calc_second_order=True)
    print(f"  Total evaluations: {X_sobol.shape[0]}  |  PCs to analyse: {n_pc}")

    sobol_results = []

    for i in range(n_pc):
        print(f"  PC {i+1}/{n_pc}...", end="  ", flush=True)

        Y  = emulators[i].predict(X_sobol)
        Si = sobol.analyze(problem, Y,
                           calc_second_order=True,
                           print_to_console=False)

        sobol_results.append({
            "S1":      Si["S1"],
            "S1_conf": Si["S1_conf"],
            "ST":      Si["ST"],
            "ST_conf": Si["ST_conf"],
            "S2":      Si.get("S2"),
            "S2_conf": Si.get("S2_conf"),
        })

        top = np.argsort(Si["ST"])[::-1][:3]
        top_str = ", ".join(
            f"{MODEL_PARAM_NAMES_SHORT[j]}={Si['ST'][j]:.3f}" for j in top
        )
        print(f"top-3: {top_str}")

    # Save per-correction pickle
    out_pkl = Path(save_folder) / "sobol_indices_global_v0.pkl"
    with open(out_pkl, "wb") as f:
        pickle.dump({
            "sobol_results": sobol_results,
            "parameter_names": param_names,
            "problem": problem,
            "correction": correction_name
        }, f)
    print(f"  ✓ Saved: {out_pkl}")

    return sobol_results

# =============================================================================
# PLOTTING
# =============================================================================

def plot_sobol_heatmap(sobol_results, param_names, save_folder, correction_name):
    ST    = np.array([Si["ST"] for Si in sobol_results])
    n_pcs = ST.shape[0]

    fig, ax = plt.subplots(figsize=(14, max(4, n_pcs * 0.8)))
    im = ax.imshow(ST, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_xticks(range(len(param_names)))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_pcs))
    ax.set_yticklabels([f"PC {i+1}" for i in range(n_pcs)], fontsize=10)

    # Annotate cells
    for r in range(n_pcs):
        for c in range(len(param_names)):
            ax.text(c, r, f"{ST[r, c]:.2f}", ha='center', va='center',
                    fontsize=7, color='black' if ST[r, c] < 0.6 else 'white')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Total-order Sobol Index ($S_T$)", fontsize=11)
    ax.set_title(f"Sobol $S_T$ Heatmap — {correction_name} — Global $v_0$",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "sobol_heatmap_global_v0.pdf")


def plot_sobol_bars_by_pc(sobol_results, save_folder, correction_name):
    n_pcs = len(sobol_results)
    ncols = min(2, n_pcs)
    nrows = int(np.ceil(n_pcs / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, (Si, ax) in enumerate(zip(sobol_results, axes)):
        ST  = Si["ST"];  S1 = Si["S1"]
        idx = np.argsort(ST)[::-1][:10]
        y   = np.arange(len(idx))

        ax.barh(y - 0.2, ST[idx], 0.35, label='$S_T$',
                color='coral', alpha=0.85)
        ax.barh(y + 0.2, S1[idx], 0.35, label='$S_1$',
                color='steelblue', alpha=0.85)

        ax.set_yticks(y)
        ax.set_yticklabels([MODEL_PARAM_NAMES_SHORT[j] for j in idx], fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Sobol Index', fontsize=10)
        ax.set_title(f'PC {i+1}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        ax.set_xlim(0, 1)

    for j in range(len(sobol_results), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'Sobol Indices by PC — {correction_name} — Global $v_0$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "sobol_bars_by_pc_global_v0.pdf")


def plot_top_parameters(sobol_results, save_folder, correction_name, top_n=10):
    ST_mean = np.mean([Si["ST"] for Si in sobol_results], axis=0)
    ST_std  = np.std( [Si["ST"] for Si in sobol_results], axis=0)
    idx     = np.argsort(ST_mean)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(np.arange(top_n), ST_mean[idx], xerr=ST_std[idx],
            color='teal', alpha=0.75, capsize=5)
    ax.set_yticks(np.arange(top_n))
    ax.set_yticklabels([MODEL_PARAM_NAMES_SHORT[i] for i in idx], fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Average $S_T$ (over PCs)", fontsize=11)
    ax.set_title(f"Top {top_n} Parameters — {correction_name} — Global $v_0$",
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "top_parameters_global_v0.pdf")


def plot_first_vs_total(sobol_results, save_folder, correction_name):
    S1 = np.mean([Si["S1"] for Si in sobol_results], axis=0)
    ST = np.mean([Si["ST"] for Si in sobol_results], axis=0)

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(S1, ST, s=80, alpha=0.7, color='steelblue', zorder=3)

    for i, name in enumerate(MODEL_PARAM_NAMES_SHORT):
        if ST[i] > 0.05:
            ax.annotate(name, (S1[i], ST[i]),
                        fontsize=8, alpha=0.85,
                        xytext=(5, 5), textcoords='offset points')

    lim = max(ST.max(), S1.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='$S_1 = S_T$ (no interactions)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('$S_1$ (First-order)', fontsize=12)
    ax.set_ylabel('$S_T$ (Total-order)', fontsize=12)
    ax.set_title(f'$S_1$ vs $S_T$ — {correction_name} — Global $v_0$',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "first_vs_total_order_global_v0.pdf")


def plot_interaction_strength(sobol_results, save_folder, correction_name):
    n_pcs   = len(sobol_results)
    n_params = len(sobol_results[0]["S1"])
    inter   = np.array([Si["ST"] - Si["S1"] for Si in sobol_results])

    fig, ax = plt.subplots(figsize=(14, max(4, n_pcs * 0.8)))
    im = ax.imshow(inter, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=0.5)

    ax.set_xticks(range(n_params))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_pcs))
    ax.set_yticklabels([f"PC {i+1}" for i in range(n_pcs)], fontsize=10)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Interaction Strength ($S_T - S_1$)", fontsize=11)
    ax.set_title(f"Interaction Strength — {correction_name} — Global $v_0$",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "interaction_strength_global_v0.pdf")


def compare_corrections_gsa(all_gsa_results, save_folder):
    corrections = list(all_gsa_results.keys())
    n_params    = len(all_gsa_results[corrections[0]][0]["ST"])

    ST_mat = np.array([
        np.mean([Si["ST"] for Si in all_gsa_results[c]], axis=0)
        for c in corrections
    ])

    fig, ax = plt.subplots(figsize=(14, max(4, len(corrections) * 1.2)))
    im = ax.imshow(ST_mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_xticks(range(n_params))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(corrections)))
    ax.set_yticklabels(corrections, fontsize=11)

    # Annotate
    for r, c in np.ndindex(ST_mat.shape):
        ax.text(c, r, f"{ST_mat[r, c]:.2f}", ha='center', va='center',
                fontsize=7, color='black' if ST_mat[r, c] < 0.6 else 'white')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Average $S_T$", fontsize=11)
    ax.set_title("GSA Comparison — All Corrections — Global $v_0$",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "gsa_comparison_all_corrections_global_v0.pdf")


def plot_top_parameters_comparison(all_gsa_results, save_folder, top_n=8):
    corrections = list(all_gsa_results.keys())
    ST_means = {
        c: np.mean([Si["ST"] for Si in all_gsa_results[c]], axis=0)
        for c in corrections
    }
    ST_global  = np.mean(list(ST_means.values()), axis=0)
    top_params = np.argsort(ST_global)[::-1][:top_n]

    n_corr = len(corrections)
    width  = 0.8 / n_corr
    colors = plt.cm.viridis(np.linspace(0, 1, n_corr))
    x      = np.arange(top_n)

    fig, ax = plt.subplots(figsize=(11, 6))
    for i, (corr, color) in enumerate(zip(corrections, colors)):
        offset = (i - n_corr / 2 + 0.5) * width
        ax.bar(x + offset, ST_means[corr][top_params], width,
               label=corr, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_PARAM_NAMES_SHORT[j] for j in top_params],
                       rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('Average $S_T$', fontsize=11)
    ax.set_title(f'Top {top_n} Parameters — All Corrections — Global $v_0$',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "top_parameters_comparison_global_v0.pdf")

# =============================================================================
# MAIN LOOP
# =============================================================================

all_gsa_results = {}

for corr in VISCOUS_CORRECTIONS:
    print("\n" + "="*80)
    print(f"GSA — {corr} — Global v0")
    print("="*80)

    corr_out = Path(OUTPUT_FOLDER) / corr
    corr_out.mkdir(parents=True, exist_ok=True)

    # --- Load emulators ---
    emulator_file = Path(EMULATOR_FOLDER) / f"gp_emulators_global_v0_{corr}.pkl"
    if not emulator_file.exists():
        print(f"  ✗ Emulator not found: {emulator_file} — skipping")
        continue

    with open(emulator_file, "rb") as f:
        emulators = pickle.load(f)
    print(f"  ✓ Loaded {len(emulators)} emulators")

    # --- Verify bad points are absent from training data ---
    pca_res      = pca_results[corr]
    idf          = CORRECTION_TO_IDF[corr]
    bad_pts      = build_bad_points_set(idf)
    dp_ids       = set(pca_res['design_point_ids'])
    overlap      = dp_ids & bad_pts
    removed_list = pca_res.get('removed_design_points', [])

    if overlap:
        print(f"  ⚠️  WARNING: {len(overlap)} bad points still in training data: "
              f"{sorted(overlap)}")
    else:
        print(f"  ✓ Bad points correctly removed ({len(removed_list)} removed: "
              f"{removed_list})")

    # --- Safe PC count ---
    n_pc = safe_n_pc(emulators, pca_res, N_PC_ANALYZE)
    print(f"  Analysing {n_pc} PC(s)")

    # --- Sobol analysis ---
    sobol_results = perform_sobol_analysis(
        emulators[:n_pc],
        n_pc,
        design_min, design_max,
        MODEL_PARAM_NAMES,
        N_SOBOL_SAMPLES,
        corr_out,
        corr,
    )
    all_gsa_results[corr] = sobol_results

    # --- Per-correction plots ---
    print(f"  Creating plots...")
    plot_sobol_heatmap(sobol_results, MODEL_PARAM_NAMES, corr_out, corr)
    plot_sobol_bars_by_pc(sobol_results, corr_out, corr)
    plot_top_parameters(sobol_results, corr_out, corr)
    plot_first_vs_total(sobol_results, corr_out, corr)
    plot_interaction_strength(sobol_results, corr_out, corr)

    print(f"  ✓ Completed {corr}")

# =============================================================================
# CROSS-CORRECTION COMPARISON
# =============================================================================

if len(all_gsa_results) > 1:
    print("\n" + "="*80)
    print("COMPARISON PLOTS")
    compare_corrections_gsa(all_gsa_results, OUTPUT_FOLDER)
    plot_top_parameters_comparison(all_gsa_results, OUTPUT_FOLDER)

# =============================================================================
# SAVE COMBINED RESULTS
# =============================================================================

print("\n" + "="*80)
print("SAVING COMBINED RESULTS")
combined_file = Path(OUTPUT_FOLDER) / "all_gsa_results_global_v0.pkl"
with open(combined_file, "wb") as f:
    pickle.dump({
        "gsa_results":           all_gsa_results,
        "parameter_names":       MODEL_PARAM_NAMES,
        "parameter_names_short": MODEL_PARAM_NAMES_SHORT,
        "n_sobol_samples":       N_SOBOL_SAMPLES,
        "corrections":           VISCOUS_CORRECTIONS,
    }, f)
print(f"  ✓ {combined_file}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SUMMARY — GSA GLOBAL v0")
print("="*80)

for corr, sobol_results in all_gsa_results.items():
    ST_mean = np.mean([Si["ST"] for Si in sobol_results], axis=0)
    top_idx = np.argsort(ST_mean)[::-1][:5]
    print(f"\n{corr}  ({len(sobol_results)} PCs):")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank}. {MODEL_PARAM_NAMES_SHORT[idx]:<15s}  ST={ST_mean[idx]:.4f}")

print("\n" + "="*80)
print("✓ Done")
print(f"  Results: {OUTPUT_FOLDER}/")
print("="*80 + "\n")


GLOBAL SENSITIVITY ANALYSIS FOR GLOBAL v0

Loading design parameter ranges...
  ✓ 17 parameters

Loading PCA results...
  ✓ PCA results loaded:
    • Grad: 3 PCs, 485 DPs used (15 removed)
    • CE: 3 PCs, 485 DPs used (15 removed)
    • PTM: 3 PCs, 475 DPs used (25 removed)
    • PTB: 3 PCs, 476 DPs used (24 removed)

GSA — Grad — Global v0
  ✓ Loaded 7 emulators
  ✓ Bad points correctly removed (15 removed: [123, 242, 289, 324, 326, 334, 341, 377, 406, 429, 440, 447, 459, 462, 483])
  Analysing 3 PC(s)
  Generating Saltelli samples (N=1024)...


/tmp/ipykernel_10493/381209697.py:154: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  X_sobol = saltelli.sample(problem, n_samples, calc_second_order=True)


  Total evaluations: 36864  |  PCs to analyse: 3
  PC 1/3...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: σ_k=0.525, w=0.279, α=0.238
  PC 2/3...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


... (23 outputs left out)

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: w=0.442, σ_k=0.316, τ_R=0.195
  PC 3/3...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: σ_k=0.269, w=0.246, τ_R=0.236
  ✓ Saved: GSA_global_v0/PTB/sobol_indices_global_v0.pkl
  Creating plots...
    Saved: GSA_global_v0/PTB/sobol_heatmap_global_v0.pdf
    Saved: GSA_global_v0/PTB/sobol_bars_by_pc_global_v0.pdf
    Saved: GSA_global_v0/PTB/top_parameters_global_v0.pdf
    Saved: GSA_global_v0/PTB/first_vs_total_order_global_v0.pdf
    Saved: GSA_global_v0/PTB/interaction_strength_global_v0.pdf
  ✓ Completed PTB

COMPARISON PLOTS
    Saved: GSA_global_v0/gsa_comparison_all_corrections_global_v0.pdf
    Saved: GSA_global_v0/top_parameters_comparison_global_v0.pdf

SAVING COMBINED RESULTS
  ✓ GSA_global_v0/all_gsa_results_global_v0.pkl

SUMMARY — GSA GLOBAL v0

Grad  (3 PCs):
  1. σ_k              ST=0.3549
  2. w                ST=0.2904
  3. τ_R              ST=0.2242
  4. α                ST=0.2072
  5. b_π              ST=0.0821

CE  (3 PCs):
  1. σ_k              ST=0.3457
  2. w                ST=0.2752
  3. τ_R              ST=0.2275
  4. α                ST=0.2

# Emulators + LOOCV for $v_0(p_T)$

In [ ]:
#!/usr/bin/env python3
"""
Emulator training + LOO validation for v0(pT).
Skips corrections whose emulator file already exists (RETRAIN = False).
"""

import os
import time
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

# =============================================================================
# CONFIGURATION
# =============================================================================

PCA_RESULTS_FILE = "pca_results/pca_results_v0pt_all_corrections.pkl"
DESIGN_FILE      = "design_pts_Pb_Pb_2760_production/design_points_main_PbPb-2760.dat"
RANGE_FILE       = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"

EMULATOR_FOLDER   = "emulators_v0pt"
VALIDATION_FOLDER = "emulator_validation_v0pt"

N_PC_TRAIN = None   # None = all PCs
RETRAIN    = False  # True forces retraining even if pkl exists

# ------------------------------------------------------------------
# BAD DESIGN POINTS
# ------------------------------------------------------------------
NAN_SETS_BY_IDF = {
    0: {334, 341, 377, 429, 447, 483},
    1: {285, 334, 341, 447, 483, 495},
    2: {209, 280, 322, 334, 341, 412, 421, 424, 429, 432, 446, 447, 453, 468, 483, 495},
    3: {60, 232, 280, 285, 322, 324, 341, 377, 432, 447, 464, 468, 482, 483, 485, 495},
}
UNFINISHED_EVENTS_SET = {289, 324, 326, 459, 462, 242, 406, 440, 123}
STRANGE_FEATURES_SET  = {289, 324, 440, 459, 462}
CORRECTION_TO_IDF     = {'Grad': 0, 'CE': 1, 'PTM': 2, 'PTB': 3}

# ------------------------------------------------------------------
# FIGURE settings
# ------------------------------------------------------------------
FIGURE_FORMAT = 'pdf'
FIGURE_DPI    = 300

# =============================================================================
# UTILITY
# =============================================================================

def save_fig(fig, filename, folder, fmt=FIGURE_FORMAT, dpi=FIGURE_DPI):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    if fmt == 'both':
        for ext in ('pdf', 'png'):
            path = folder / f'{stem}.{ext}'
            fig.savefig(path, dpi=dpi, bbox_inches='tight')
            print(f"    Saved: {path}")
    else:
        path = folder / f'{stem}.{fmt}'
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print(f"    Saved: {path}")
    plt.close(fig)


def build_bad_points_set(idf):
    return NAN_SETS_BY_IDF[idf] | UNFINISHED_EVENTS_SET | STRANGE_FEATURES_SET


def align_design_to_pca(design_df, design_point_ids):
    return design_df.iloc[design_point_ids].values


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan


def safe_n_pc(emulators, pca_res, n_pc_train):
    return min(
        len(emulators),
        pca_res['n_components'],
        n_pc_train if n_pc_train is not None else 999
    )

# =============================================================================
# LOAD DATA
# =============================================================================

print("\n" + "="*80)
print("EMULATOR TRAINING FOR v0(pT)  —  with LOO validation")
print("="*80)

print("\nLoading design points...")
design_df     = pd.read_csv(DESIGN_FILE, index_col=0)
design_range  = pd.read_csv(RANGE_FILE,  index_col=0)
design_max    = design_range["max"].values
design_min    = design_range["min"].values
design_labels = list(design_df.columns)
print(f"  ✓ {design_df.shape[0]} design points, {design_df.shape[1]} parameters")

print("\nLoading PCA results...")
if not Path(PCA_RESULTS_FILE).exists():
    raise FileNotFoundError(
        f"PCA file not found: {PCA_RESULTS_FILE}\n"
        "Run the v0(pT) PCA script first."
    )

with open(PCA_RESULTS_FILE, "rb") as f:
    pca_results_dict = pickle.load(f)

print("  ✓ PCA results loaded:")
for k, v in pca_results_dict.items():
    n_removed = len(v.get('removed_design_points', []))
    obs       = v['Y_data'].shape[1]
    print(f"    • {k}: {v['n_components']} PCs, "
          f"{len(v['design_point_ids'])} DPs, "
          f"{obs} observables "
          f"({n_removed} bad DPs removed)")

Path(EMULATOR_FOLDER).mkdir(parents=True, exist_ok=True)
Path(VALIDATION_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# KERNEL BUILDER
# =============================================================================

def make_kernel(X):
    ptp = np.ptp(X, axis=0)
    ptp[ptp == 0] = 1.0
    return (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=ptp,
              length_scale_bounds=np.outer(ptp, (0.4, 1e2)))
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-2, 1e2))
    )

# =============================================================================
# TRAINING
# =============================================================================

def train_emulators(X, pc_scores, n_pc_train=None,
                    emulator_file=None, retrain=False):
    if n_pc_train is None:
        n_pc_train = pc_scores.shape[1]
    n_pc_train = min(n_pc_train, pc_scores.shape[1])

    if emulator_file and Path(emulator_file).exists() and not retrain:
        print(f"  Loading existing emulators from {emulator_file}")
        with open(emulator_file, "rb") as f:
            emulators = pickle.load(f)
        return emulators[:n_pc_train]

    print(f"  Training {n_pc_train} GP emulators...")
    emulators = []
    for i in range(n_pc_train):
        t0  = time.time()
        gpr = GaussianProcessRegressor(
            kernel=make_kernel(X),
            n_restarts_optimizer=5,
            alpha=1e-10,
            normalize_y=True,
        )
        gpr.fit(X, pc_scores[:, i])
        r2 = gpr.score(X, pc_scores[:, i])
        print(f"    PC {i+1:02d}: R²={r2:.4f}  ({time.time()-t0:.1f}s)")
        emulators.append(gpr)

    if emulator_file:
        with open(emulator_file, "wb") as f:
            pickle.dump(emulators, f)
        print(f"  Saved → {emulator_file}")

    return emulators

# =============================================================================
# LOO VALIDATION  (fast analytical)
# =============================================================================

def loo_validate_fast(X, pc_scores, emulators):
    n    = X.shape[0]
    n_pc = min(len(emulators), pc_scores.shape[1])

    y_true = pc_scores[:, :n_pc].copy()
    y_pred = np.zeros_like(y_true)
    y_std  = np.zeros_like(y_true)

    for i, gpr in enumerate(emulators[:n_pc]):
        K       = gpr.kernel_(X)
        noise   = np.exp(gpr.kernel_.theta[-1])
        K_noisy = K + noise * np.eye(n)

        L     = np.linalg.cholesky(K_noisy)
        L_inv = np.linalg.solve(L, np.eye(n))
        K_inv = L_inv.T @ L_inv

        y = y_true[:, i]
        if gpr.normalize_y:
            y_mean = y.mean()
            y_std_ = y.std() or 1.0
            y_norm = (y - y_mean) / y_std_
        else:
            y_norm = y
            y_mean, y_std_ = 0.0, 1.0

        diag         = K_inv.diagonal()
        loo_norm     = y_norm - (K_inv @ y_norm) / (diag + 1e-12)
        loo_var      = 1.0 / (diag + 1e-12)

        y_pred[:, i] = loo_norm * y_std_ + y_mean
        y_std[:, i]  = np.sqrt(np.abs(loo_var)) * y_std_

    return y_true, y_pred, y_std


def loo_r2_list(y_true, y_pred):
    return [r2_score(y_true[:, i], y_pred[:, i])
            for i in range(y_true.shape[1])]

# =============================================================================
# RECONSTRUCTION: PC scores → v0(pT)
# =============================================================================

def reconstruct_from_pc_scores(pc_scores_subset, inverse_tf_matrix, scaler):
    n    = pc_scores_subset.shape[0]
    n_pc = pc_scores_subset.shape[1]
    itf  = inverse_tf_matrix[:n_pc, :]
    Y_sc = pc_scores_subset @ itf / math.sqrt(n - 1)
    return scaler.inverse_transform(Y_sc)

# =============================================================================
# PLOTS
# =============================================================================

def plot_loo_per_pc(y_true, y_pred, y_std, correction_name, save_folder):
    n_pc  = y_true.shape[1]
    ncols = min(4, n_pc)
    nrows = math.ceil(n_pc / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.5 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i in range(n_pc):
        ax = axes[i]
        yt = y_true[:, i];  yp = y_pred[:, i];  ys = y_std[:, i]

        z  = np.abs((yt - yp) / (ys + 1e-12))
        sc = ax.scatter(yt, yp, c=z, cmap='RdYlGn_r',
                        vmin=0, vmax=3, s=18, alpha=0.7)
        plt.colorbar(sc, ax=ax, label='|z|')

        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'k--', lw=1.5)

        r2   = r2_score(yt, yp)
        rmse = np.sqrt(np.mean((yt - yp) ** 2))
        c68  = np.mean(np.abs(yt - yp) < ys) * 100
        c95  = np.mean(np.abs(yt - yp) < 2 * ys) * 100

        ax.set_title(f"PC {i+1} | R²={r2:.3f}  RMSE={rmse:.3f}",
                     fontsize=10, fontweight='bold')
        ax.set_xlabel("True PC score", fontsize=9)
        ax.set_ylabel("LOO predicted", fontsize=9)
        ax.text(0.04, 0.96,
                f"68% cov: {c68:.0f}%\n95% cov: {c95:.0f}%",
                transform=ax.transAxes, fontsize=8,
                va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
        ax.grid(True, alpha=0.3)

    for j in range(n_pc, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'LOO Validation — {correction_name} — $v_0(p_T)$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'loo_validation_v0pt_{correction_name}', save_folder)


def plot_loo_residuals(y_true, y_pred, y_std, correction_name, save_folder):
    n_pc  = y_true.shape[1]
    ncols = min(4, n_pc)
    nrows = math.ceil(n_pc / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.0 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    x_ref = np.linspace(-5, 5, 300)
    gauss = np.exp(-0.5 * x_ref**2) / np.sqrt(2 * np.pi)

    for i in range(n_pc):
        ax = axes[i]
        z  = (y_true[:, i] - y_pred[:, i]) / (y_std[:, i] + 1e-12)

        ax.hist(z, bins=30, density=True, color='steelblue',
                alpha=0.6, label='LOO pulls')
        ax.plot(x_ref, gauss, 'r-', lw=2, label='N(0,1)')
        ax.set_xlim(-5, 5)
        ax.set_xlabel(r'Pull $(y - \hat{y})\,/\,\sigma$', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'PC {i+1}  |  '
                     f'$\\mu$={z.mean():.2f}  $\\sigma$={z.std():.2f}',
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for j in range(n_pc, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'LOO Pull Distributions — {correction_name} — $v_0(p_T)$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'loo_pulls_v0pt_{correction_name}', save_folder)


def plot_reconstruction_loo(Y_orig, Y_loo, observable_labels,
                            correction_name, save_folder, n_examples=3):
    n_examples   = min(n_examples, Y_orig.shape[0])
    centralities = sorted({lbl[0] for lbl in observable_labels})
    colors       = plt.cm.viridis(np.linspace(0, 1, len(centralities)))

    fig, axes = plt.subplots(n_examples, 2,
                             figsize=(14, 4.5 * n_examples))
    if n_examples == 1:
        axes = axes[np.newaxis, :]

    for row in range(n_examples):
        orig  = Y_orig[row]
        recon = Y_loo[row]
        ax0, ax1 = axes[row]

        for cent_idx, color in zip(centralities, colors):
            mask    = [lbl[0] == cent_idx for lbl in observable_labels]
            pt_vals = [lbl[1] for lbl, m in zip(observable_labels, mask) if m]

            ax0.plot(pt_vals, orig[mask],  'o-',  color=color,
                     label=f'Cent {cent_idx}', alpha=0.8, markersize=4)
            ax0.plot(pt_vals, recon[mask], 's--', color=color,
                     alpha=0.6, markersize=3)

            rel_err = np.abs((orig[mask] - recon[mask]) /
                             np.where(np.abs(orig[mask]) > 1e-6,
                                      orig[mask], np.nan))
            ax1.plot(pt_vals, rel_err, 'o-', color=color,
                     label=f'Cent {cent_idx}', alpha=0.8, markersize=3)

        for ax in (ax0, ax1):
            ax.set_xlabel(r'$p_T$ [GeV/c]', fontsize=10)
            ax.set_xscale('log')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7, ncol=2)

        ax0.set_ylabel(r'$v_0(p_T)$', fontsize=10)
        ax0.set_title(f'DP {row}: original (solid) vs LOO (dashed)', fontsize=10)
        ax1.set_ylabel('Relative Error', fontsize=10)
        ax1.set_title(f'DP {row}: LOO reconstruction error', fontsize=10)

    fig.suptitle(f'LOO Reconstruction — {correction_name} — $v_0(p_T)$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'loo_reconstruction_v0pt_{correction_name}', save_folder)


def plot_summary(all_emulators, all_loo, pca_results_dict,
                 all_design_matrices, save_folder):
    corrections = list(all_emulators.keys())
    colors      = plt.cm.tab10(np.linspace(0, 1, len(corrections)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for corr, color in zip(corrections, colors):
        emulators = all_emulators[corr]
        X         = all_design_matrices[corr]
        pc_scores = pca_results_dict[corr]['pc_scores']
        n_pc      = min(len(emulators), pc_scores.shape[1])

        r2_in = [r2_score(pc_scores[:, i], emulators[i].predict(X))
                 for i in range(n_pc)]
        axes[0].plot(range(1, n_pc + 1), r2_in, 'o-',
                     color=color, label=corr, lw=2, markersize=6)

        r2_loo = loo_r2_list(all_loo[corr]['y_true'],
                              all_loo[corr]['y_pred'])
        axes[1].plot(range(1, len(r2_loo) + 1), r2_loo, 'o-',
                     color=color, label=corr, lw=2, markersize=6)

    for ax, title in zip(axes, ['In-sample R²', 'LOO R²']):
        ax.set_xlabel('Principal Component', fontsize=12)
        ax.set_ylabel('R²', fontsize=12)
        ax.set_title(f'{title} — $v_0(p_T)$', fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(0.9, color='gray', ls=':', lw=1.5)
        ax.set_ylim(0.3, 1.02)

    plt.tight_layout()
    save_fig(fig, 'emulator_summary_v0pt', save_folder)

# =============================================================================
# MAIN LOOP
# =============================================================================

all_emulators       = {}
all_loo             = {}
all_design_matrices = {}

for corr, pca_res in pca_results_dict.items():
    print("\n" + "="*80)
    print(f"Viscous correction: {corr}")
    print("="*80)

    pc_scores         = pca_res['pc_scores']
    design_point_ids  = pca_res['design_point_ids']
    observable_labels = pca_res['observable_labels']
    n_pc_total        = pca_res['n_components']
    scaler            = pca_res['scaler']
    inverse_tf_matrix = pca_res['inverse_tf_matrix']
    Y_orig            = pca_res['Y_data']

    X = align_design_to_pca(design_df, design_point_ids)
    all_design_matrices[corr] = X

    emulator_file = f"{EMULATOR_FOLDER}/gp_emulators_v0pt_{corr}.pkl"
    n_pc_train    = N_PC_TRAIN if N_PC_TRAIN else n_pc_total

    # ------------------------------------------------------------------
    # FAST PATH: emulator already exists and RETRAIN=False
    # ------------------------------------------------------------------
    if Path(emulator_file).exists() and not RETRAIN:
        print(f"  ✓ Emulator file found — loading (skipping training)")
        with open(emulator_file, "rb") as f:
            emulators = pickle.load(f)
        emulators = emulators[:safe_n_pc(emulators, pca_res, n_pc_train)]
        all_emulators[corr] = emulators

        print(f"  Running LOO validation...")
        y_true, y_pred, y_std = loo_validate_fast(X, pc_scores, emulators)
        all_loo[corr] = {'y_true': y_true, 'y_pred': y_pred, 'y_std': y_std}

        n_pc = y_true.shape[1]
        print(f"\n   {'PC':>4}  {'R²(LOO)':>9}  {'RMSE':>8}  "
              f"{'68% cov':>9}  {'95% cov':>9}")
        print(f"   {'-'*4}  {'-'*9}  {'-'*8}  {'-'*9}  {'-'*9}")
        for i in range(n_pc):
            yt = y_true[:, i]; yp = y_pred[:, i]; ys = y_std[:, i]
            r2   = r2_score(yt, yp)
            rmse = np.sqrt(np.mean((yt - yp) ** 2))
            c68  = np.mean(np.abs(yt - yp) < ys) * 100
            c95  = np.mean(np.abs(yt - yp) < 2 * ys) * 100
            print(f"   {i+1:>4}  {r2:>9.4f}  {rmse:>8.4f}  "
                  f"{c68:>8.1f}%  {c95:>8.1f}%")

        # Reconstruction check
        Y_loo = reconstruct_from_pc_scores(y_pred, inverse_tf_matrix, scaler)
        threshold  = 1e-4
        mask_valid = np.abs(Y_orig) > threshold
        if mask_valid.any():
            rel_err = np.abs((Y_orig - Y_loo)[mask_valid] / Y_orig[mask_valid])
            print(f"\n   LOO reconstruction — mean rel. error: {rel_err.mean():.4f}")
            print(f"   LOO reconstruction — max  rel. error: {rel_err.max():.4f}")

        # Plots
        print(f"\n  Saving validation plots...")
        corr_val_folder = f"{VALIDATION_FOLDER}/{corr}"
        plot_loo_per_pc(y_true, y_pred, y_std, corr, corr_val_folder)
        plot_loo_residuals(y_true, y_pred, y_std, corr, corr_val_folder)
        plot_reconstruction_loo(Y_orig, Y_loo, observable_labels,
                                corr, corr_val_folder, n_examples=3)

        print(f"\n✓ Finished {corr} (loaded from disk)")
        continue   # ← skip full training block below

    # ------------------------------------------------------------------
    # FULL PATH: train from scratch
    # ------------------------------------------------------------------
    print(f"  Design matrix shape  : {X.shape}")
    print(f"  PC scores shape      : {pc_scores.shape}")
    print(f"  Observables (pT bins): {Y_orig.shape[1]}")
    print(f"  pT range kept        : "
          f"{min(lbl[1] for lbl in observable_labels):.3f} – "
          f"{max(lbl[1] for lbl in observable_labels):.3f} GeV/c")
    print(f"  Centralities kept    : "
          f"{sorted({lbl[0] for lbl in observable_labels})}")

    idf     = CORRECTION_TO_IDF[corr]
    bad_pts = build_bad_points_set(idf)
    overlap = set(design_point_ids) & bad_pts
    removed = pca_res.get('removed_design_points', [])
    if overlap:
        print(f"  ⚠️  WARNING: {len(overlap)} bad points still present: "
              f"{sorted(overlap)}")
    else:
        print(f"  ✓ Bad points removed ({len(removed)}): {removed}")

    # 1. Train
    print(f"\n1. Training emulators...")
    emulators = train_emulators(
        X, pc_scores,
        n_pc_train    = n_pc_train,
        emulator_file = emulator_file,
        retrain       = RETRAIN,
    )
    all_emulators[corr] = emulators

    # 2. LOO
    print(f"\n2. LOO validation (fast analytical)...")
    y_true, y_pred, y_std = loo_validate_fast(X, pc_scores, emulators)
    all_loo[corr] = {'y_true': y_true, 'y_pred': y_pred, 'y_std': y_std}

    n_pc = y_true.shape[1]
    print(f"\n   {'PC':>4}  {'R²(LOO)':>9}  {'RMSE':>8}  "
          f"{'68% cov':>9}  {'95% cov':>9}")
    print(f"   {'-'*4}  {'-'*9}  {'-'*8}  {'-'*9}  {'-'*9}")
    for i in range(n_pc):
        yt = y_true[:, i]; yp = y_pred[:, i]; ys = y_std[:, i]
        r2   = r2_score(yt, yp)
        rmse = np.sqrt(np.mean((yt - yp) ** 2))
        c68  = np.mean(np.abs(yt - yp) < ys) * 100
        c95  = np.mean(np.abs(yt - yp) < 2 * ys) * 100
        print(f"   {i+1:>4}  {r2:>9.4f}  {rmse:>8.4f}  "
              f"{c68:>8.1f}%  {c95:>8.1f}%")

    # 3. Reconstruction
    print(f"\n3. Computing LOO reconstruction in v0(pT) space...")
    Y_loo = reconstruct_from_pc_scores(y_pred, inverse_tf_matrix, scaler)
    threshold  = 1e-4
    mask_valid = np.abs(Y_orig) > threshold
    if mask_valid.any():
        rel_err = np.abs((Y_orig - Y_loo)[mask_valid] / Y_orig[mask_valid])
        print(f"   LOO reconstruction — mean rel. error: {rel_err.mean():.4f}")
        print(f"   LOO reconstruction — max  rel. error: {rel_err.max():.4f}")

    # 4. Plots
    print(f"\n4. Saving validation plots...")
    corr_val_folder = f"{VALIDATION_FOLDER}/{corr}"
    plot_loo_per_pc(y_true, y_pred, y_std, corr, corr_val_folder)
    plot_loo_residuals(y_true, y_pred, y_std, corr, corr_val_folder)
    plot_reconstruction_loo(Y_orig, Y_loo, observable_labels,
                            corr, corr_val_folder, n_examples=3)

    print(f"\n✓ Finished {corr}")

# =============================================================================
# SUMMARY PLOT
# =============================================================================

print("\n" + "="*80)
print("SUMMARY PLOT")
plot_summary(all_emulators, all_loo, pca_results_dict,
             all_design_matrices, VALIDATION_FOLDER)

# =============================================================================
# SAVE ALL EMULATORS
# =============================================================================

print("\n" + "="*80)
print("SAVING ALL EMULATORS")
emulator_file_all = f"{EMULATOR_FOLDER}/all_gp_emulators_v0pt.pkl"
with open(emulator_file_all, "wb") as f:
    pickle.dump(all_emulators, f)
print(f"  ✓ {emulator_file_all}")

# =============================================================================
# FINAL SUMMARY TABLE
# =============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY — v0(pT)")
print("="*80)
print(f"\n{'Correction':<10} {'PCs':>5} {'DPs':>6} {'Obs':>5} "
      f"{'min R²(in)':>12} {'min R²(LOO)':>13} {'avg R²(LOO)':>13}")
print("-" * 72)

for corr in all_emulators:
    emulators = all_emulators[corr]
    X         = all_design_matrices[corr]
    pc_scores = pca_results_dict[corr]['pc_scores']
    n_pc      = min(len(emulators), pc_scores.shape[1])
    obs       = pca_results_dict[corr]['Y_data'].shape[1]

    r2_in  = [r2_score(pc_scores[:, i], emulators[i].predict(X))
               for i in range(n_pc)]
    r2_loo = loo_r2_list(all_loo[corr]['y_true'],
                          all_loo[corr]['y_pred'])

    print(f"{corr:<10} {n_pc:>5} {X.shape[0]:>6} {obs:>5} "
          f"{min(r2_in):>12.4f} {min(r2_loo):>13.4f} "
          f"{np.nanmean(r2_loo):>13.4f}")

print("\n" + "="*80)
print("✓ Done")
print(f"  Emulators : {EMULATOR_FOLDER}/")
print(f"  Validation: {VALIDATION_FOLDER}/")
print("="*80 + "\n")


EMULATOR TRAINING FOR v0(pT)  —  with LOO validation

Loading design points...
  ✓ 500 design points, 17 parameters

Loading PCA results...
  ✓ PCA results loaded:
    • Grad: 9 PCs, 485 DPs, 91 observables (15 bad DPs removed)
    • CE: 9 PCs, 485 DPs, 91 observables (15 bad DPs removed)
    • PTM: 9 PCs, 475 DPs, 91 observables (25 bad DPs removed)
    • PTB: 11 PCs, 476 DPs, 91 observables (24 bad DPs removed)

Viscous correction: Grad
  ✓ Emulator file found — loading (skipping training)
  Running LOO validation...

     PC    R²(LOO)      RMSE    68% cov    95% cov
   ----  ---------  --------  ---------  ---------
      1     0.9474    0.2292      81.4%      97.9%
      2     0.9529    0.2168      80.2%      96.9%
      3     0.7452    0.5043      83.7%      98.4%
      4     0.5933    0.6371      82.5%      99.0%
      5     0.1310    0.9312      82.5%      99.2%
      6     0.0929    0.9514      81.9%      97.9%
      7     0.0543    0.9715      81.6%      99.2%
      8     0.

# Global Sensitivity Analysis (GSA) for $v_0(p_T)$

In [1]:
#!/usr/bin/env python3
"""
Global Sobol Sensitivity Analysis for v0(pT) — all viscous corrections.
"""

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from SALib.sample import saltelli
from SALib.analyze import sobol

# =============================================================================
# CONFIGURATION
# =============================================================================

PCA_RESULTS_FILE = "pca_results/pca_results_v0pt_all_corrections.pkl"
EMULATOR_FOLDER  = "emulators_v0pt"
OUTPUT_FOLDER    = "GSA_v0pt"

DESIGN_FILE = "design_pts_Pb_Pb_2760_production/design_points_main_PbPb-2760.dat"
RANGE_FILE  = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"

N_SOBOL_SAMPLES = 1024
N_PC_ANALYZE    = None   # None = all PCs saved in emulator file

VISCOUS_CORRECTIONS = ["Grad", "CE", "PTM", "PTB"]

# ------------------------------------------------------------------
# BAD DESIGN POINTS
# ------------------------------------------------------------------
NAN_SETS_BY_IDF = {
    0: {334, 341, 377, 429, 447, 483},
    1: {285, 334, 341, 447, 483, 495},
    2: {209, 280, 322, 334, 341, 412, 421, 424, 429, 432, 446, 447, 453, 468, 483, 495},
    3: {60, 232, 280, 285, 322, 324, 341, 377, 432, 447, 464, 468, 482, 483, 485, 495},
}
UNFINISHED_EVENTS_SET = {289, 324, 326, 459, 462, 242, 406, 440, 123}
STRANGE_FEATURES_SET  = {289, 324, 440, 459, 462}
CORRECTION_TO_IDF     = {'Grad': 0, 'CE': 1, 'PTM': 2, 'PTB': 3}

# ------------------------------------------------------------------
# FIGURE settings
# ------------------------------------------------------------------
FIGURE_FORMAT = 'pdf'
FIGURE_DPI    = 300

# =============================================================================
# PARAMETER NAMES
# =============================================================================

MODEL_PARAM_NAMES = [
    '$N$[$2.76$TeV]', '$p$', '$\\sigma_k$', '$w$ [fm]',
    '$d_{\\mathrm{min}}$ [fm]', '$\\tau_R$ [fm/$c$]', '$\\alpha$',
    '$T_{\\eta,\\mathrm{kink}}$ [GeV]',
    '$a_{\\eta,\\mathrm{low}}$ [GeV${}^{-1}$]',
    '$a_{\\eta,\\mathrm{high}}$ [GeV${}^{-1}$]',
    '$(\\eta/s)_{\\mathrm{kink}}$', '$(\\zeta/s)_{\\max}$',
    '$T_{\\zeta,c}$ [GeV]', '$w_{\\zeta}$ [GeV]',
    '$\\lambda_{\\zeta}$', '$b_{\\pi}$', '$T_{\\mathrm{sw}}$ [GeV]'
]

MODEL_PARAM_NAMES_SHORT = [
    'N', 'p', 'σ_k', 'w', 'd_min', 'τ_R', 'α',
    'T_η,kink', 'a_η,low', 'a_η,high', '(η/s)_kink',
    '(ζ/s)_max', 'T_ζ,c', 'w_ζ', 'λ_ζ', 'b_π', 'T_sw'
]

# =============================================================================
# UTILITY
# =============================================================================

def save_fig(fig, path, fmt=FIGURE_FORMAT, dpi=FIGURE_DPI):
    stem   = Path(path).stem
    folder = Path(path).parent
    if fmt == 'both':
        for ext in ('pdf', 'png'):
            p = folder / f'{stem}.{ext}'
            fig.savefig(p, dpi=dpi, bbox_inches='tight')
            print(f"    Saved: {p}")
    else:
        p = folder / f'{stem}.{fmt}'
        fig.savefig(p, dpi=dpi, bbox_inches='tight')
        print(f"    Saved: {p}")
    plt.close(fig)


def build_bad_points_set(idf):
    return NAN_SETS_BY_IDF[idf] | UNFINISHED_EVENTS_SET | STRANGE_FEATURES_SET


def safe_n_pc(emulators, pca_res, n_pc_analyze):
    return min(
        len(emulators),
        pca_res['n_components'],
        n_pc_analyze if n_pc_analyze is not None else 999
    )

# =============================================================================
# LOAD DATA
# =============================================================================

print("\n" + "="*80)
print("GLOBAL SENSITIVITY ANALYSIS FOR v0(pT)")
print("="*80)

print("\nLoading design parameter ranges...")
design_df    = pd.read_csv(DESIGN_FILE, index_col=0)
design_range = pd.read_csv(RANGE_FILE,  index_col=0)
design_max   = design_range["max"].values
design_min   = design_range["min"].values
print(f"  ✓ {len(design_min)} parameters")

print("\nLoading PCA results...")
if not Path(PCA_RESULTS_FILE).exists():
    raise FileNotFoundError(f"PCA file not found: {PCA_RESULTS_FILE}")

with open(PCA_RESULTS_FILE, "rb") as f:
    pca_results = pickle.load(f)

print("  ✓ PCA results loaded:")
for k, v in pca_results.items():
    n_removed = len(v.get('removed_design_points', []))
    obs       = v['Y_data'].shape[1]
    print(f"    • {k}: {v['n_components']} PCs, "
          f"{len(v['design_point_ids'])} DPs, "
          f"{obs} observables "
          f"({n_removed} bad DPs removed)")

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# SOBOL ANALYSIS
# =============================================================================

def perform_sobol_analysis(emulators, n_pc, design_min, design_max,
                           param_names, n_samples, save_folder, correction_name):
    n_params = len(design_min)

    problem = {
        "num_vars": n_params,
        "names":    param_names,
        "bounds":   list(zip(design_min, design_max))
    }

    print(f"  Generating Saltelli samples (N={n_samples})...")
    X_sobol = saltelli.sample(problem, n_samples, calc_second_order=True)
    print(f"  Total evaluations: {X_sobol.shape[0]}  |  PCs: {n_pc}")

    sobol_results = []

    for i in range(n_pc):
        print(f"  PC {i+1}/{n_pc}...", end="  ", flush=True)

        Y  = emulators[i].predict(X_sobol)
        Si = sobol.analyze(problem, Y,
                           calc_second_order=True,
                           print_to_console=False)

        sobol_results.append({
            "S1":      Si["S1"],
            "S1_conf": Si["S1_conf"],
            "ST":      Si["ST"],
            "ST_conf": Si["ST_conf"],
            "S2":      Si.get("S2"),
            "S2_conf": Si.get("S2_conf"),
        })

        top     = np.argsort(Si["ST"])[::-1][:3]
        top_str = ", ".join(
            f"{MODEL_PARAM_NAMES_SHORT[j]}={Si['ST'][j]:.3f}" for j in top
        )
        print(f"top-3: {top_str}")

    out_pkl = Path(save_folder) / "sobol_indices_v0pt.pkl"
    with open(out_pkl, "wb") as f:
        pickle.dump({
            "sobol_results":   sobol_results,
            "parameter_names": param_names,
            "problem":         problem,
            "correction":      correction_name,
        }, f)
    print(f"  ✓ Saved: {out_pkl}")

    return sobol_results

# =============================================================================
# PLOTTING
# =============================================================================

def plot_sobol_heatmap(sobol_results, save_folder, correction_name):
    ST    = np.array([Si["ST"] for Si in sobol_results])
    n_pcs = ST.shape[0]

    fig, ax = plt.subplots(figsize=(14, max(4, n_pcs * 0.8)))
    im = ax.imshow(ST, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_xticks(range(len(MODEL_PARAM_NAMES_SHORT)))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_pcs))
    ax.set_yticklabels([f"PC {i+1}" for i in range(n_pcs)], fontsize=10)

    for r in range(n_pcs):
        for c in range(ST.shape[1]):
            ax.text(c, r, f"{ST[r, c]:.2f}", ha='center', va='center',
                    fontsize=7, color='black' if ST[r, c] < 0.6 else 'white')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Total-order Sobol Index ($S_T$)", fontsize=11)
    ax.set_title(f"Sobol $S_T$ Heatmap — {correction_name} — $v_0(p_T)$",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "sobol_heatmap_v0pt.pdf")


def plot_sobol_bars_by_pc(sobol_results, save_folder, correction_name):
    n_pcs = len(sobol_results)
    ncols = min(3, n_pcs)
    nrows = int(np.ceil(n_pcs / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, (Si, ax) in enumerate(zip(sobol_results, axes)):
        ST  = Si["ST"];  S1 = Si["S1"]
        idx = np.argsort(ST)[::-1][:10]
        y   = np.arange(len(idx))

        ax.barh(y - 0.2, ST[idx], 0.35, label='$S_T$',
                color='coral', alpha=0.85)
        ax.barh(y + 0.2, S1[idx], 0.35, label='$S_1$',
                color='steelblue', alpha=0.85)

        ax.set_yticks(y)
        ax.set_yticklabels([MODEL_PARAM_NAMES_SHORT[j] for j in idx], fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Sobol Index', fontsize=10)
        ax.set_title(f'PC {i+1}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        ax.set_xlim(0, 1)

    for j in range(len(sobol_results), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f'Sobol Indices by PC — {correction_name} — $v_0(p_T)$',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "sobol_bars_by_pc_v0pt.pdf")


def plot_top_parameters(sobol_results, save_folder, correction_name, top_n=10):
    ST_mean = np.mean([Si["ST"] for Si in sobol_results], axis=0)
    ST_std  = np.std( [Si["ST"] for Si in sobol_results], axis=0)
    idx     = np.argsort(ST_mean)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(np.arange(top_n), ST_mean[idx], xerr=ST_std[idx],
            color='teal', alpha=0.75, capsize=5)
    ax.set_yticks(np.arange(top_n))
    ax.set_yticklabels([MODEL_PARAM_NAMES_SHORT[i] for i in idx], fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Average $S_T$ (over PCs)", fontsize=11)
    ax.set_title(f"Top {top_n} Parameters — {correction_name} — $v_0(p_T)$",
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "top_parameters_v0pt.pdf")


def plot_first_vs_total(sobol_results, save_folder, correction_name):
    S1 = np.mean([Si["S1"] for Si in sobol_results], axis=0)
    ST = np.mean([Si["ST"] for Si in sobol_results], axis=0)

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(S1, ST, s=80, alpha=0.7, color='steelblue', zorder=3)

    for i, name in enumerate(MODEL_PARAM_NAMES_SHORT):
        if ST[i] > 0.05:
            ax.annotate(name, (S1[i], ST[i]),
                        fontsize=8, alpha=0.85,
                        xytext=(5, 5), textcoords='offset points')

    lim = max(ST.max(), S1.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'r--', lw=1.5,
            label='$S_1 = S_T$ (no interactions)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('$S_1$ (First-order)', fontsize=12)
    ax.set_ylabel('$S_T$ (Total-order)', fontsize=12)
    ax.set_title(f'$S_1$ vs $S_T$ — {correction_name} — $v_0(p_T)$',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "first_vs_total_order_v0pt.pdf")


def plot_interaction_strength(sobol_results, save_folder, correction_name):
    n_pcs  = len(sobol_results)
    inter  = np.array([Si["ST"] - Si["S1"] for Si in sobol_results])

    fig, ax = plt.subplots(figsize=(14, max(4, n_pcs * 0.8)))
    im = ax.imshow(inter, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=0.5)

    ax.set_xticks(range(inter.shape[1]))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_pcs))
    ax.set_yticklabels([f"PC {i+1}" for i in range(n_pcs)], fontsize=10)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Interaction Strength ($S_T - S_1$)", fontsize=11)
    ax.set_title(f"Interaction Strength — {correction_name} — $v_0(p_T)$",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "interaction_strength_v0pt.pdf")


def plot_st_by_centrality(sobol_results, pca_res, save_folder, correction_name):
    """
    Weight S_T of each PC by the fraction of variance it explains in each
    centrality bin, giving an effective S_T per centrality.
    Uses the inverse_tf_matrix rows as proxy for per-centrality importance.
    """
    observable_labels = pca_res['observable_labels']
    inv_tf            = pca_res['inverse_tf_matrix']   # (n_pc, n_obs)
    n_pc              = len(sobol_results)

    centralities = sorted({lbl[0] for lbl in observable_labels})
    n_cent       = len(centralities)
    n_params     = len(sobol_results[0]["ST"])

    # Weight: squared loading of each PC in each centrality bin, normalised
    ST_cent = np.zeros((n_cent, n_params))

    for ci, cent_idx in enumerate(centralities):
        mask    = np.array([lbl[0] == cent_idx for lbl in observable_labels])
        weights = np.array([
            np.sum(inv_tf[i, mask] ** 2) for i in range(n_pc)
        ])
        weights /= weights.sum() + 1e-12

        for i, Si in enumerate(sobol_results[:n_pc]):
            ST_cent[ci] += weights[i] * Si["ST"]

    fig, ax = plt.subplots(figsize=(14, max(4, n_cent * 0.8)))
    im = ax.imshow(ST_cent, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_xticks(range(n_params))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_cent))
    ax.set_yticklabels([f"{c*10}–{(c+1)*10}%" for c in centralities], fontsize=10)

    for r in range(n_cent):
        for c in range(n_params):
            ax.text(c, r, f"{ST_cent[r, c]:.2f}", ha='center', va='center',
                    fontsize=7, color='black' if ST_cent[r, c] < 0.6 else 'white')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Effective $S_T$ per centrality", fontsize=11)
    ax.set_title(f"$S_T$ by Centrality — {correction_name} — $v_0(p_T)$",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "sobol_by_centrality_v0pt.pdf")


def compare_corrections_gsa(all_gsa_results, save_folder):
    corrections = list(all_gsa_results.keys())
    n_params    = len(all_gsa_results[corrections[0]][0]["ST"])

    ST_mat = np.array([
        np.mean([Si["ST"] for Si in all_gsa_results[c]], axis=0)
        for c in corrections
    ])

    fig, ax = plt.subplots(figsize=(14, max(4, len(corrections) * 1.2)))
    im = ax.imshow(ST_mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_xticks(range(n_params))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(corrections)))
    ax.set_yticklabels(corrections, fontsize=11)

    for r, c in np.ndindex(ST_mat.shape):
        ax.text(c, r, f"{ST_mat[r, c]:.2f}", ha='center', va='center',
                fontsize=7, color='black' if ST_mat[r, c] < 0.6 else 'white')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Average $S_T$", fontsize=11)
    ax.set_title("GSA Comparison — All Corrections — $v_0(p_T)$",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "gsa_comparison_all_corrections_v0pt.pdf")


def plot_top_parameters_comparison(all_gsa_results, save_folder, top_n=8):
    corrections = list(all_gsa_results.keys())
    ST_means = {
        c: np.mean([Si["ST"] for Si in all_gsa_results[c]], axis=0)
        for c in corrections
    }
    ST_global  = np.mean(list(ST_means.values()), axis=0)
    top_params = np.argsort(ST_global)[::-1][:top_n]

    n_corr = len(corrections)
    width  = 0.8 / n_corr
    colors = plt.cm.viridis(np.linspace(0, 1, n_corr))
    x      = np.arange(top_n)

    fig, ax = plt.subplots(figsize=(11, 6))
    for i, (corr, color) in enumerate(zip(corrections, colors)):
        offset = (i - n_corr / 2 + 0.5) * width
        ax.bar(x + offset, ST_means[corr][top_params], width,
               label=corr, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_PARAM_NAMES_SHORT[j] for j in top_params],
                       rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('Average $S_T$', fontsize=11)
    ax.set_title(f'Top {top_n} Parameters — All Corrections — $v_0(p_T)$',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / "top_parameters_comparison_v0pt.pdf")

# =============================================================================
# MAIN LOOP
# =============================================================================

all_gsa_results = {}

for corr in VISCOUS_CORRECTIONS:
    print("\n" + "="*80)
    print(f"GSA — {corr} — v0(pT)")
    print("="*80)

    corr_out = Path(OUTPUT_FOLDER) / corr
    corr_out.mkdir(parents=True, exist_ok=True)

    # Load emulators
    emulator_file = Path(EMULATOR_FOLDER) / f"gp_emulators_v0pt_{corr}.pkl"
    if not emulator_file.exists():
        print(f"  ✗ Emulator not found: {emulator_file} — skipping")
        continue

    with open(emulator_file, "rb") as f:
        emulators = pickle.load(f)
    print(f"  ✓ Loaded {len(emulators)} emulators")

    # Verify bad points removed from training
    pca_res      = pca_results[corr]
    idf          = CORRECTION_TO_IDF[corr]
    bad_pts      = build_bad_points_set(idf)
    dp_ids       = set(pca_res['design_point_ids'])
    overlap      = dp_ids & bad_pts
    removed_list = pca_res.get('removed_design_points', [])

    if overlap:
        print(f"  ⚠️  WARNING: {len(overlap)} bad points in training data: "
              f"{sorted(overlap)}")
    else:
        print(f"  ✓ Bad points correctly removed "
              f"({len(removed_list)}): {removed_list}")

    # Safe PC count
    n_pc = safe_n_pc(emulators, pca_res, N_PC_ANALYZE)
    print(f"  Analysing {n_pc} PC(s)  |  "
          f"{pca_res['Y_data'].shape[1]} observables  |  "
          f"pT: {min(lbl[1] for lbl in pca_res['observable_labels']):.3f}–"
          f"{max(lbl[1] for lbl in pca_res['observable_labels']):.3f} GeV/c")

    # Sobol analysis
    sobol_results = perform_sobol_analysis(
        emulators[:n_pc],
        n_pc,
        design_min, design_max,
        MODEL_PARAM_NAMES,
        N_SOBOL_SAMPLES,
        corr_out,
        corr,
    )
    all_gsa_results[corr] = sobol_results

    # Per-correction plots
    print(f"  Creating plots...")
    plot_sobol_heatmap(sobol_results, corr_out, corr)
    plot_sobol_bars_by_pc(sobol_results, corr_out, corr)
    plot_top_parameters(sobol_results, corr_out, corr)
    plot_first_vs_total(sobol_results, corr_out, corr)
    plot_interaction_strength(sobol_results, corr_out, corr)
    plot_st_by_centrality(sobol_results, pca_res, corr_out, corr)

    print(f"  ✓ Completed {corr}")

# =============================================================================
# CROSS-CORRECTION COMPARISON
# =============================================================================

if len(all_gsa_results) > 1:
    print("\n" + "="*80)
    print("COMPARISON PLOTS")
    compare_corrections_gsa(all_gsa_results, OUTPUT_FOLDER)
    plot_top_parameters_comparison(all_gsa_results, OUTPUT_FOLDER)

# =============================================================================
# SAVE COMBINED RESULTS
# =============================================================================

print("\n" + "="*80)
print("SAVING COMBINED RESULTS")
combined_file = Path(OUTPUT_FOLDER) / "all_gsa_results_v0pt.pkl"
with open(combined_file, "wb") as f:
    pickle.dump({
        "gsa_results":           all_gsa_results,
        "parameter_names":       MODEL_PARAM_NAMES,
        "parameter_names_short": MODEL_PARAM_NAMES_SHORT,
        "n_sobol_samples":       N_SOBOL_SAMPLES,
        "corrections":           VISCOUS_CORRECTIONS,
    }, f)
print(f"  ✓ {combined_file}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SUMMARY — GSA v0(pT)")
print("="*80)

for corr, sobol_results in all_gsa_results.items():
    ST_mean = np.mean([Si["ST"] for Si in sobol_results], axis=0)
    top_idx = np.argsort(ST_mean)[::-1][:5]
    print(f"\n{corr}  ({len(sobol_results)} PCs):")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank}. {MODEL_PARAM_NAMES_SHORT[idx]:<15s}  ST={ST_mean[idx]:.4f}")

print("\n" + "="*80)
print("✓ Done")
print(f"  Results: {OUTPUT_FOLDER}/")
print("="*80 + "\n")


GLOBAL SENSITIVITY ANALYSIS FOR v0(pT)

Loading design parameter ranges...
  ✓ 17 parameters

Loading PCA results...
  ✓ PCA results loaded:
    • Grad: 9 PCs, 485 DPs, 91 observables (15 bad DPs removed)
    • CE: 9 PCs, 485 DPs, 91 observables (15 bad DPs removed)
    • PTM: 9 PCs, 475 DPs, 91 observables (25 bad DPs removed)
    • PTB: 11 PCs, 476 DPs, 91 observables (24 bad DPs removed)

GSA — Grad — v0(pT)
  ✓ Loaded 9 emulators
  ✓ Bad points correctly removed (15): [123, 242, 289, 324, 326, 334, 341, 377, 406, 429, 440, 447, 459, 462, 483]
  Analysing 9 PC(s)  |  91 observables  |  pT: 0.552–2.494 GeV/c
  Generating Saltelli samples (N=1024)...


/tmp/ipykernel_21419/828969998.py:150: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  X_sobol = saltelli.sample(problem, n_samples, calc_second_order=True)


  Total evaluations: 36864  |  PCs: 9
  PC 1/9...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: σ_k=0.627, τ_R=0.105, α=0.082
  PC 2/9...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


... (75 outputs left out)

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: T_ζ,c=0.388, (ζ/s)_max=0.334, τ_R=0.324
  PC 11/11...  

/sysroot/home/rafaela/.local/lib/python3.9/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))


top-3: σ_k=0.466, T_η,kink=0.326, w_ζ=0.262
  ✓ Saved: GSA_v0pt/PTB/sobol_indices_v0pt.pkl
  Creating plots...
    Saved: GSA_v0pt/PTB/sobol_heatmap_v0pt.pdf
    Saved: GSA_v0pt/PTB/sobol_bars_by_pc_v0pt.pdf
    Saved: GSA_v0pt/PTB/top_parameters_v0pt.pdf
    Saved: GSA_v0pt/PTB/first_vs_total_order_v0pt.pdf
    Saved: GSA_v0pt/PTB/interaction_strength_v0pt.pdf
    Saved: GSA_v0pt/PTB/sobol_by_centrality_v0pt.pdf
  ✓ Completed PTB

COMPARISON PLOTS
    Saved: GSA_v0pt/gsa_comparison_all_corrections_v0pt.pdf
    Saved: GSA_v0pt/top_parameters_comparison_v0pt.pdf

SAVING COMBINED RESULTS
  ✓ GSA_v0pt/all_gsa_results_v0pt.pkl

SUMMARY — GSA v0(pT)

Grad  (9 PCs):
  1. w                ST=0.2358
  2. σ_k              ST=0.1798
  3. τ_R              ST=0.1518
  4. α                ST=0.1412
  5. T_sw             ST=0.1052

CE  (9 PCs):
  1. w                ST=0.2282
  2. τ_R              ST=0.2083
  3. σ_k              ST=0.2045
  4. α                ST=0.1698
  5. (ζ/s)_max        ST=0.12

# Global Sensitivity Analysis (GSA) for $v_0(p_T)/v_0$

In [1]:
#!/usr/bin/env python3
"""
GSA for v0(pT), global v0, and their ratio v0(pT)/v0.
Single script: Sobol indices + three-observable comparison plots.
"""

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path

# SALib — compatible with both < 1.4 and >= 1.4
try:
    from SALib.sample.sobol import sample as saltelli_sample
    from SALib.analyze.sobol import analyze as sobol_analyze
except ImportError:
    from SALib.sample.saltelli import sample as saltelli_sample
    from SALib.analyze.sobol import analyze as sobol_analyze

# =============================================================================
# CONFIGURATION
# =============================================================================

PCA_RESULTS_V0PT_FILE = "pca_results/pca_results_v0pt_all_corrections.pkl"
PCA_RESULTS_V0_FILE   = "pca_results/pca_results_global_v0_all_corrections.pkl"
EMULATOR_FOLDER_V0PT  = "emulators_v0pt"
EMULATOR_FOLDER_V0    = "emulators_global_v0"

DESIGN_FILE = "design_pts_Pb_Pb_2760_production/design_points_main_PbPb-2760.dat"
RANGE_FILE  = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"

OUTPUT_FOLDER = "GSA_ratio"

VISCOUS_CORRECTIONS = ["Grad", "CE", "PTM", "PTB"]

N_SOBOL_SAMPLES   = 1024
N_PC_ANALYZE_V0PT = None
N_PC_ANALYZE_V0   = None

CENTRALITIES_TO_PLOT = [0, 1, 2, 3, 4, 5, 6]

# ------------------------------------------------------------------
# BAD DESIGN POINTS
# ------------------------------------------------------------------
NAN_SETS_BY_IDF = {
    0: {334, 341, 377, 429, 447, 483},
    1: {285, 334, 341, 447, 483, 495},
    2: {209, 280, 322, 334, 341, 412, 421, 424, 429, 432, 446, 447, 453, 468, 483, 495},
    3: {60, 232, 280, 285, 322, 324, 341, 377, 432, 447, 464, 468, 482, 483, 485, 495},
}
UNFINISHED_EVENTS_SET = {289, 324, 326, 459, 462, 242, 406, 440, 123}
STRANGE_FEATURES_SET  = {289, 324, 440, 459, 462}
CORRECTION_TO_IDF     = {'Grad': 0, 'CE': 1, 'PTM': 2, 'PTB': 3}

# ------------------------------------------------------------------
# FIGURE settings
# ------------------------------------------------------------------
FIGURE_FORMAT = 'pdf'
FIGURE_DPI    = 300

# =============================================================================
# PARAMETER NAMES
# =============================================================================

MODEL_PARAM_NAMES = [
    '$N$[$2.76$TeV]', '$p$', '$\\sigma_k$', '$w$ [fm]',
    '$d_{\\mathrm{min}}$ [fm]', '$\\tau_R$ [fm/$c$]', '$\\alpha$',
    '$T_{\\eta,\\mathrm{kink}}$ [GeV]',
    '$a_{\\eta,\\mathrm{low}}$ [GeV${}^{-1}$]',
    '$a_{\\eta,\\mathrm{high}}$ [GeV${}^{-1}$]',
    '$(\\eta/s)_{\\mathrm{kink}}$', '$(\\zeta/s)_{\\max}$',
    '$T_{\\zeta,c}$ [GeV]', '$w_{\\zeta}$ [GeV]',
    '$\\lambda_{\\zeta}$', '$b_{\\pi}$', '$T_{\\mathrm{sw}}$ [GeV]'
]
MODEL_PARAM_NAMES_SHORT = [
    'N', 'p', 'σ_k', 'w', 'd_min', 'τ_R', 'α',
    'T_η,kink', 'a_η,low', 'a_η,high', '(η/s)_kink',
    '(ζ/s)_max', 'T_ζ,c', 'w_ζ', 'λ_ζ', 'b_π', 'T_sw'
]
N_PARAMS = len(MODEL_PARAM_NAMES)

OBS_KEYS = ["v0", "v0pt", "ratio"]
OBS_COLORS = {
    'v0':    '#2166ac',
    'v0pt':  '#d6604d',
    'ratio': '#1a9850',
}
OBS_LABELS = {
    'v0':    r'$v_0^{\rm global}$',
    'v0pt':  r'$v_0(p_T)$',
    'ratio': r'$v_0(p_T)\,/\,v_0^{\rm global}$',
}

# =============================================================================
# UTILITY
# =============================================================================

def save_fig(fig, path, fmt=FIGURE_FORMAT, dpi=FIGURE_DPI):
    stem   = Path(path).stem
    folder = Path(path).parent
    folder.mkdir(parents=True, exist_ok=True)
    if fmt == 'both':
        for ext in ('pdf', 'png'):
            p = folder / f'{stem}.{ext}'
            fig.savefig(p, dpi=dpi, bbox_inches='tight')
            print(f"    Saved: {p}")
    else:
        p = folder / f'{stem}.{fmt}'
        fig.savefig(p, dpi=dpi, bbox_inches='tight')
        print(f"    Saved: {p}")
    plt.close(fig)


def build_bad_points_set(idf):
    return NAN_SETS_BY_IDF[idf] | UNFINISHED_EVENTS_SET | STRANGE_FEATURES_SET


def safe_n_pc(emulators, pca_res, n_pc_max):
    return min(
        len(emulators),
        pca_res['n_components'],
        n_pc_max if n_pc_max is not None else 999
    )


def cent_label(cent_idx):
    return f"{cent_idx*10}–{(cent_idx+1)*10}%"

# =============================================================================
# LOAD DATA
# =============================================================================

print("\n" + "="*80)
print("GSA FOR v0(pT), GLOBAL v0, AND RATIO v0(pT)/v0")
print("="*80)

print("\nLoading parameter ranges...")
design_df    = pd.read_csv(DESIGN_FILE, index_col=0)
design_range = pd.read_csv(RANGE_FILE,  index_col=0)
design_max   = design_range["max"].values
design_min   = design_range["min"].values
print(f"  ✓ {N_PARAMS} parameters")

print("\nLoading PCA results...")
with open(PCA_RESULTS_V0PT_FILE, "rb") as f:
    pca_v0pt = pickle.load(f)
with open(PCA_RESULTS_V0_FILE, "rb") as f:
    pca_v0 = pickle.load(f)

for k in pca_v0pt:
    print(f"  v0(pT) {k}: {pca_v0pt[k]['n_components']} PCs, "
          f"{pca_v0pt[k]['Y_data'].shape[1]} obs")
for k in pca_v0:
    print(f"  v0     {k}: {pca_v0[k]['n_components']} PCs, "
          f"{pca_v0[k]['Y_data'].shape[1]} obs")

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# PREDICTION HELPERS
# =============================================================================

def predict_pc_scores(emulators, X_sobol):
    return np.column_stack([emu.predict(X_sobol) for emu in emulators])


def pc_to_observable(pc_scores, inverse_tf_matrix, scaler):
    import math
    n, n_pc = pc_scores.shape
    itf     = inverse_tf_matrix[:n_pc, :]
    Y_std   = pc_scores @ itf / math.sqrt(n - 1)
    return scaler.inverse_transform(Y_std)

# =============================================================================
# SOBOL ANALYSIS — all three observables from one Saltelli draw
# =============================================================================

def sobol_all_observables(emulators_v0pt, emulators_v0,
                          pca_res_v0pt, pca_res_v0,
                          design_min, design_max, n_samples):

    problem = {
        "num_vars": N_PARAMS,
        "names":    MODEL_PARAM_NAMES,
        "bounds":   list(zip(design_min, design_max))
    }

    print(f"    Saltelli sampling (N={n_samples})...", end=" ", flush=True)
    X = saltelli_sample(problem, n_samples, calc_second_order=False)
    print(f"→ {X.shape[0]} evaluations")

    inv_tf_v0pt = pca_res_v0pt['inverse_tf_matrix']
    inv_tf_v0   = pca_res_v0['inverse_tf_matrix']
    sc_v0pt     = pca_res_v0pt['scaler']
    sc_v0       = pca_res_v0['scaler']
    obs_v0pt    = pca_res_v0pt['observable_labels']
    obs_v0      = pca_res_v0['observable_labels']

    pc_v0pt = predict_pc_scores(emulators_v0pt, X)
    pc_v0   = predict_pc_scores(emulators_v0,   X)

    Y_v0pt = pc_to_observable(pc_v0pt, inv_tf_v0pt, sc_v0pt)
    Y_v0   = pc_to_observable(pc_v0,   inv_tf_v0,   sc_v0)

    # --- diagnostics ---
    cents_v0pt = sorted({lbl[0] for lbl in obs_v0pt})
    cents_v0   = sorted({lbl[0] for lbl in obs_v0})
    print(f"    Centralities v0(pT): {cents_v0pt}")
    print(f"    Centralities v0    : {cents_v0}")

    # --- centrality lookup: cent_idx → first column in Y_v0 ---
    v0_cent_to_col = {}
    for col_idx, lbl in enumerate(obs_v0):
        cent_idx = lbl[0] if isinstance(lbl, (tuple, list)) else int(lbl[0])
        if cent_idx not in v0_cent_to_col:
            v0_cent_to_col[cent_idx] = col_idx
    print(f"    v0 cent→col map    : {v0_cent_to_col}")

    # --- ratio columns ---
    ratio_cols = []
    for col_idx, lbl in enumerate(obs_v0pt):
        cent_idx = lbl[0] if isinstance(lbl, (tuple, list)) else int(lbl[0])
        if cent_idx in v0_cent_to_col:
            v0_col = v0_cent_to_col[cent_idx]
            with np.errstate(divide='ignore', invalid='ignore'):
                r = np.where(
                    np.abs(Y_v0[:, v0_col]) > 1e-8,
                    Y_v0pt[:, col_idx] / Y_v0[:, v0_col],
                    np.nan
                )
            ratio_cols.append(r)

    if not ratio_cols:
        raise ValueError(
            f"No centrality overlap between v0(pT) {cents_v0pt} "
            f"and v0 global {cents_v0}.\n"
            "Check that CENTRALITIES_TO_INCLUDE matches in both PCA scripts."
        )

    Y_ratio = np.column_stack(ratio_cols)
    print(f"    Y_v0pt shape : {Y_v0pt.shape}")
    print(f"    Y_v0   shape : {Y_v0.shape}")
    print(f"    Y_ratio shape: {Y_ratio.shape}")

    # --- Sobol analysis per observable column, then average ---
    def analyse_matrix(Y_mat):
        ST_all = np.zeros((Y_mat.shape[1], N_PARAMS))
        S1_all = np.zeros((Y_mat.shape[1], N_PARAMS))
        for j in range(Y_mat.shape[1]):
            col = Y_mat[:, j]
            if np.all(np.isnan(col)) or np.nanstd(col) < 1e-12:
                continue
            col = np.nan_to_num(col, nan=np.nanmean(col))
            Si  = sobol_analyze(problem, col,
                                calc_second_order=False,
                                print_to_console=False)
            ST_all[j] = np.clip(Si["ST"], 0, None)
            S1_all[j] = np.clip(Si["S1"], 0, None)
        return {
            "ST":        ST_all.mean(axis=0),
            "S1":        S1_all.mean(axis=0),
            "ST_by_obs": ST_all,
            "S1_by_obs": S1_all,
        }

    print("    Sobol: v0(pT) ...", end=" ", flush=True)
    res_v0pt  = analyse_matrix(Y_v0pt)
    print("done  |  v0 ...", end=" ", flush=True)
    res_v0    = analyse_matrix(Y_v0)
    print("done  |  ratio ...", end=" ", flush=True)
    res_ratio = analyse_matrix(Y_ratio)
    print("done")

    return {
        "v0pt":  res_v0pt,
        "v0":    res_v0,
        "ratio": res_ratio,
        "obs_labels_v0pt": obs_v0pt,
        "obs_labels_v0":   obs_v0,
    }

# =============================================================================
# PLOTTING — per correction
# =============================================================================

def plot_three_obs_heatmap(gsa_res, correction_name, save_folder):
    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    for ax, key in zip(axes, OBS_KEYS):
        ST = gsa_res[key]["ST"].reshape(1, -1)
        im = ax.imshow(ST, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
        ax.set_yticks([0])
        ax.set_yticklabels([OBS_LABELS[key]], fontsize=11)
        ax.set_xticks([])
        for c in range(N_PARAMS):
            ax.text(c, 0, f"{ST[0, c]:.2f}", ha='center', va='center',
                    fontsize=7,
                    color='black' if ST[0, c] < 0.6 else 'white')
        plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
    axes[-1].set_xticks(range(N_PARAMS))
    axes[-1].set_xticklabels(MODEL_PARAM_NAMES_SHORT,
                             rotation=45, ha='right', fontsize=9)
    fig.suptitle(f"Average $S_T$ — {correction_name}",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) /
             f"gsa_three_obs_heatmap_{correction_name}.pdf")


def plot_three_obs_bars(gsa_res, correction_name, save_folder, top_n=10):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
    for ax, key in zip(axes, OBS_KEYS):
        ST  = gsa_res[key]["ST"]
        S1  = gsa_res[key]["S1"]
        idx = np.argsort(ST)[::-1][:top_n]
        y   = np.arange(len(idx))
        ax.barh(y - 0.2, ST[idx], 0.35,
                label='$S_T$', color='coral', alpha=0.85)
        ax.barh(y + 0.2, S1[idx], 0.35,
                label='$S_1$', color='steelblue', alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels([MODEL_PARAM_NAMES_SHORT[i] for i in idx], fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Sobol Index', fontsize=10)
        ax.set_title(OBS_LABELS[key], fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xlim(0, 1)
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(f'Sobol Indices by Observable — {correction_name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) /
             f"gsa_three_obs_bars_{correction_name}.pdf")


def plot_st_by_centrality_ratio(gsa_res, correction_name, save_folder):
    obs_labels = gsa_res["obs_labels_v0pt"]
    ST_by_obs  = gsa_res["ratio"]["ST_by_obs"]
    centralities = sorted({lbl[0] for lbl in obs_labels
                           if lbl[0] in CENTRALITIES_TO_PLOT})
    ST_cent = np.zeros((len(centralities), N_PARAMS))
    for ci, cent_idx in enumerate(centralities):
        rows = [j for j, lbl in enumerate(obs_labels) if lbl[0] == cent_idx]
        if rows:
            ST_cent[ci] = ST_by_obs[rows, :].mean(axis=0)

    fig, ax = plt.subplots(figsize=(14, max(4, len(centralities) * 0.9)))
    im = ax.imshow(ST_cent, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(N_PARAMS))
    ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT,
                       rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(centralities)))
    ax.set_yticklabels([cent_label(c) for c in centralities], fontsize=10)
    for r in range(len(centralities)):
        for c in range(N_PARAMS):
            ax.text(c, r, f"{ST_cent[r, c]:.2f}", ha='center', va='center',
                    fontsize=7,
                    color='black' if ST_cent[r, c] < 0.6 else 'white')
    plt.colorbar(im, ax=ax).set_label("$S_T$ (avg over $p_T$)", fontsize=11)
    ax.set_title(f"$S_T$ for ratio by centrality — {correction_name}",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) /
             f"gsa_ratio_by_centrality_{correction_name}.pdf")

# =============================================================================
# PLOTTING — cross-correction
# =============================================================================

def plot_cross_correction_comparison(all_gsa, save_folder):
    corrections = list(all_gsa.keys())
    for key in OBS_KEYS:
        ST_mat = np.array([all_gsa[c][key]["ST"] for c in corrections])
        fig, ax = plt.subplots(figsize=(14, max(4, len(corrections) * 1.2)))
        im = ax.imshow(ST_mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
        ax.set_xticks(range(N_PARAMS))
        ax.set_xticklabels(MODEL_PARAM_NAMES_SHORT,
                           rotation=45, ha='right', fontsize=9)
        ax.set_yticks(range(len(corrections)))
        ax.set_yticklabels(corrections, fontsize=11)
        for r, c in np.ndindex(ST_mat.shape):
            ax.text(c, r, f"{ST_mat[r, c]:.2f}", ha='center', va='center',
                    fontsize=7,
                    color='black' if ST_mat[r, c] < 0.6 else 'white')
        plt.colorbar(im, ax=ax).set_label("Average $S_T$", fontsize=11)
        ax.set_title(f"Cross-correction $S_T$ — {OBS_LABELS[key]}",
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        save_fig(fig, Path(save_folder) /
                 f"gsa_cross_correction_{key}.pdf")


def plot_top_params_comparison_three_obs(all_gsa, save_folder, top_n=8):
    corrections = list(all_gsa.keys())
    n_corr  = len(corrections)
    colors  = plt.cm.viridis(np.linspace(0, 1, n_corr))
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
    for ax, key in zip(axes, OBS_KEYS):
        ST_global  = np.mean([all_gsa[c][key]["ST"] for c in corrections], axis=0)
        top_params = np.argsort(ST_global)[::-1][:top_n]
        x     = np.arange(top_n)
        width = 0.8 / n_corr
        for i, (corr, color) in enumerate(zip(corrections, colors)):
            offset = (i - n_corr / 2 + 0.5) * width
            ax.bar(x + offset, all_gsa[corr][key]["ST"][top_params],
                   width, label=corr, color=color, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([MODEL_PARAM_NAMES_SHORT[j] for j in top_params],
                           rotation=40, ha='right', fontsize=9)
        ax.set_ylabel('$S_T$', fontsize=11)
        ax.set_title(OBS_LABELS[key], fontsize=12, fontweight='bold')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 1)
    fig.suptitle(f'Top-{top_n} Parameters by Observable — All Corrections',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, Path(save_folder) /
             "gsa_top_params_three_obs_all_corrections.pdf")


def plot_gsa_summary_all_observables(all_gsa, save_folder, top_n=10):
    """
    Single figure: S1 (left) and ST (right) for all three observables.
    Bars = mean over viscous corrections; error bars = min–max range.
    """
    corrections = list(all_gsa.keys())

    data = {}
    for key in OBS_KEYS:
        ST_mat = np.array([all_gsa[c][key]["ST"] for c in corrections])
        S1_mat = np.array([all_gsa[c][key]["S1"] for c in corrections])
        data[key] = {
            'ST_mean': ST_mat.mean(axis=0),
            'ST_min':  ST_mat.min(axis=0),
            'ST_max':  ST_mat.max(axis=0),
            'S1_mean': S1_mat.mean(axis=0),
            'S1_min':  S1_mat.min(axis=0),
            'S1_max':  S1_mat.max(axis=0),
        }

    # rank by highest ST across all observables
    ST_envelope = np.max([data[key]['ST_mean'] for key in OBS_KEYS], axis=0)
    top_idx     = np.argsort(ST_envelope)[::-1][:top_n]
    param_labels = [MODEL_PARAM_NAMES_SHORT[i] for i in top_idx]

    fig, (ax_s1, ax_st) = plt.subplots(
        1, 2, figsize=(14, 0.6 * top_n + 2.5), sharey=True
    )

    y       = np.arange(top_n)
    offsets = {'v0': -0.22, 'v0pt': 0.0, 'ratio': 0.22}
    bar_h   = 0.18

    for key in OBS_KEYS:
        d      = data[key]
        color  = OBS_COLORS[key]
        offset = offsets[key]

        for ax, mean_key, lo_key, hi_key in [
            (ax_s1, 'S1_mean', 'S1_min', 'S1_max'),
            (ax_st, 'ST_mean', 'ST_min', 'ST_max'),
        ]:
            vals   = d[mean_key][top_idx]
            err_lo = np.clip(vals - d[lo_key][top_idx], 0, None)
            err_hi = np.clip(d[hi_key][top_idx] - vals, 0, None)

            ax.barh(y + offset, vals, bar_h, color=color, alpha=0.82)
            ax.errorbar(vals, y + offset,
                        xerr=[err_lo, err_hi],
                        fmt='none', color='k', linewidth=0.9, capsize=3)

    for ax, title in [(ax_s1, 'First-order  $S_1$'),
                      (ax_st, 'Total-order  $S_T$')]:
        ax.set_yticks(y)
        ax.set_yticklabels(param_labels, fontsize=10)
        ax.invert_yaxis()
        ax.set_xlabel('Sobol Index', fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1)
        ax.axvline(0, color='k', lw=0.5)
        ax.grid(axis='x', alpha=0.3)

    legend_handles = [
        mpatches.Patch(color=OBS_COLORS[k], alpha=0.82, label=OBS_LABELS[k])
        for k in OBS_KEYS
    ]
    ax_s1.legend(handles=legend_handles, fontsize=10,
                 loc='lower right', framealpha=0.9)

    fig.suptitle(
        'Global Sensitivity Analysis — All Observables\n'
        f'Mean ± range over: {", ".join(corrections)}',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    save_fig(fig, Path(save_folder) / 'gsa_summary_all_observables.pdf')

# =============================================================================
# MAIN LOOP
# =============================================================================

all_gsa = {}

for corr in VISCOUS_CORRECTIONS:
    print("\n" + "="*80)
    print(f"GSA — {corr}")
    print("="*80)

    corr_out = Path(OUTPUT_FOLDER) / corr
    corr_out.mkdir(parents=True, exist_ok=True)

    emu_path_v0pt = Path(EMULATOR_FOLDER_V0PT) / f"gp_emulators_v0pt_{corr}.pkl"
    emu_path_v0   = Path(EMULATOR_FOLDER_V0)   / f"gp_emulators_global_v0_{corr}.pkl"

    for p in (emu_path_v0pt, emu_path_v0):
        if not p.exists():
            raise FileNotFoundError(f"Emulator not found: {p}")

    with open(emu_path_v0pt, "rb") as f:
        emulators_v0pt = pickle.load(f)
    with open(emu_path_v0, "rb") as f:
        emulators_v0 = pickle.load(f)

    pca_res_v0pt = pca_v0pt[corr]
    pca_res_v0   = pca_v0[corr]

    n_pc_v0pt = safe_n_pc(emulators_v0pt, pca_res_v0pt, N_PC_ANALYZE_V0PT)
    n_pc_v0   = safe_n_pc(emulators_v0,   pca_res_v0,   N_PC_ANALYZE_V0)
    emulators_v0pt = emulators_v0pt[:n_pc_v0pt]
    emulators_v0   = emulators_v0[:n_pc_v0]

    # verify bad points
    idf     = CORRECTION_TO_IDF[corr]
    bad_pts = build_bad_points_set(idf)
    for tag, pca_res in [("v0pt", pca_res_v0pt), ("v0", pca_res_v0)]:
        overlap = set(pca_res['design_point_ids']) & bad_pts
        removed = pca_res.get('removed_design_points', [])
        if overlap:
            print(f"  ⚠️  {tag}: {len(overlap)} bad points still present!")
        else:
            print(f"  ✓  {tag}: {len(removed)} bad points removed")

    print(f"  PCs — v0(pT): {n_pc_v0pt}  |  v0: {n_pc_v0}")
    print(f"  Observables — v0(pT): {pca_res_v0pt['Y_data'].shape[1]}  "
          f"|  v0: {pca_res_v0['Y_data'].shape[1]}")

    gsa_res = sobol_all_observables(
        emulators_v0pt, emulators_v0,
        pca_res_v0pt, pca_res_v0,
        design_min, design_max,
        N_SOBOL_SAMPLES,
    )
    all_gsa[corr] = gsa_res

    print("  Creating plots...")
    plot_three_obs_heatmap(gsa_res, corr, corr_out)
    plot_three_obs_bars(gsa_res, corr, corr_out)
    plot_st_by_centrality_ratio(gsa_res, corr, corr_out)

    out_pkl = corr_out / "gsa_three_obs.pkl"
    with open(out_pkl, "wb") as f:
        pickle.dump(gsa_res, f)
    print(f"  ✓ Saved: {out_pkl}")
    print(f"  ✓ Completed {corr}")

# =============================================================================
# CROSS-CORRECTION PLOTS
# =============================================================================

if len(all_gsa) > 1:
    print("\n" + "="*80)
    print("CROSS-CORRECTION PLOTS")
    plot_cross_correction_comparison(all_gsa, OUTPUT_FOLDER)
    plot_top_params_comparison_three_obs(all_gsa, OUTPUT_FOLDER)
    plot_gsa_summary_all_observables(all_gsa, OUTPUT_FOLDER, top_n=10)

# =============================================================================
# SAVE COMBINED RESULTS
# =============================================================================

print("\n" + "="*80)
print("SAVING COMBINED RESULTS")
combined = Path(OUTPUT_FOLDER) / "all_gsa_three_obs.pkl"
with open(combined, "wb") as f:
    pickle.dump({
        "gsa_results":           all_gsa,
        "parameter_names":       MODEL_PARAM_NAMES,
        "parameter_names_short": MODEL_PARAM_NAMES_SHORT,
        "n_sobol_samples":       N_SOBOL_SAMPLES,
        "corrections":           VISCOUS_CORRECTIONS,
    }, f)
print(f"  ✓ {combined}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

for corr in all_gsa:
    print(f"\n{corr}:")
    for key in OBS_KEYS:
        ST   = all_gsa[corr][key]["ST"]
        top3 = np.argsort(ST)[::-1][:3]
        top3_str = "  ".join(
            f"{MODEL_PARAM_NAMES_SHORT[i]}={ST[i]:.3f}" for i in top3
        )
        print(f"  {OBS_LABELS[key]:40s}  top-3: {top3_str}")

print("\n" + "="*80)
print("✓ Done")
print(f"  Results: {OUTPUT_FOLDER}/")
print("="*80 + "\n")


EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  v0(pT)/v0
Correction: Grad  |  Centrality: 50-60%
  ✓ 17 parameters, 20000 original posterior samples
  ✓ v0(pT) emulators: 9 PCs, 13 pT bins, cent 50-60%
  ✓ v0 global obs_labels: ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']  →  '50-60%' = col 5
  ✓ v0 global emulators: 3 PCs

Generating predictions...
    Prior: 2000 samples ... done  shape=(2000, 13)  valid=2000  range=[-5.426, 26.549]
    Original posterior (pT-integrated): 20000 samples ... done  shape=(20000, 13)  valid=20000  range=[-4.300, 19.099]
    Scaled Spectra (Grad): 1400 samples ... done  shape=(1400, 13)  valid=1400  range=[-5.994, 17.575]
  Spread check — prior: mean σ = 1.408221
  Spread check — original: mean σ = 0.475065
  Spread check — scaled spectra: mean σ = 0.912693

Loading experimental data...
  ✓ ATLAS 5.02 TeV v0(pT)  29 bins  pT=[0.55,9.46]
  ✓ ATLAS v0 global: 9 centrality bins
  ✓ CMS 2.76 TeV v0(pT)  